# **PINN CLÁSICA CON 6 CAPAS 64 NEURONAS**

## **1. Librerías usadas**

In [ ]:
#Starndard libreries
import os
import sys
import io
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random

#import libreries of NN
import tensorflow as tf
from tensorflow import keras
import tensorflow_probability as tfp

import tensorflow as tf
tf.keras.backend.set_floatx('float64') # Fuerza todo a 64 bits globalmente

#import optimization
from scipy.optimize import minimize
from scipy.interpolate import CubicSpline
import scipy.optimize


In [ ]:
# Define la ruta
path = '/content/ABUN_PINN'

# Crea la carpeta si no existe
if not os.path.exists(path):
    os.makedirs(path)
    print(f"Carpeta '{path}' creada con éxito.")
else:
    print(f"La carpeta '{path}' ya existe.")

## **2. Entrenamiento del la PINN**

In [ ]:
# Configurar semilla
seed = 0
tf.random.set_seed(seed)
np.random.seed(seed)
random.seed(seed)

# Parámetros físicos
sigma = 1.0
k0_sim = [1.0, 2.0, 5.0]
A_sim = [1.0, 1.0, 1.0]
k_boost = 0.0
m = 1.0
c = 1.0


#Parámetros iniciales
N = 12000
L = 80 * sigma
dx = L / N
T = L/2 - 4*sigma
dt = T / N

X_min, X_max = -L/2, L/2
T_min, T_max = 0.0, T

def condition_initial(x, sigma, k0, k_boost, m, A, dx):
    k0_arr = np.array(k0).reshape(-1, 1)
    A_arr = np.array(A).reshape(-1, 1)
    x_arr = x.reshape(1, -1)

    fase_estacionaria = np.sum(A_arr * np.exp(1j * k0_arr * x_arr) + A_arr * np.exp(-1j * k0_arr * x_arr), axis=0)
    fase_estacionaria = fase_estacionaria.reshape(-1, 1)

    psi_static = np.exp(-(x**2)/(2 * sigma**2)) * fase_estacionaria
    psi_moving = psi_static * np.exp(1j * k_boost * x)
    psi_moving = psi_moving / np.sqrt(np.max(np.abs(psi_moving)**2))

    phi0 = psi_moving.real
    phi0 = phi0 / np.sqrt(np.max(np.abs(phi0)**2))

    phi_k = np.fft.fft(psi_moving.flatten())
    k_freqs = 2 * np.pi * np.fft.fftfreq(len(x), d=dx)
    omega_k = np.sqrt(k_freqs**2 + m**2)

    pi_k_calc = -1j * omega_k * phi_k
    pi0 = np.fft.ifft(pi_k_calc).real.reshape(-1, 1)

    return phi0, pi0

# Interpolación de phi y pi
x_grid = np.linspace(X_min, X_max, 4000).reshape(-1, 1)
phi0_vals, pi0_vals = condition_initial(x_grid, sigma, k0_sim, k_boost, m, A_sim, dx)
interp_phi0 = CubicSpline(x_grid.flatten(), phi0_vals.flatten(), extrapolate=True)
interp_pi0  = CubicSpline(x_grid.flatten(), pi0_vals.flatten(), extrapolate=True)

In [ ]:
class FourierEmbedding(tf.keras.layers.Layer):
    def __init__(self, num_features=128, sigma=10.0, input_dim=2, **kwargs):
        super().__init__(**kwargs)
        self.num_features = num_features
        self.sigma = sigma
        self.input_dim = input_dim

    def build(self, input_shape):
        self.B = self.add_weight(
            name='B',
            shape=(self.input_dim, self.num_features),
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=self.sigma),
            trainable=False,
            dtype=tf.float32
        )
        super().build(input_shape)

    def call(self, x):
        x = tf.cast(x, tf.float32)                              # ← forzar float32
        projected = 2.0 * np.pi * tf.matmul(x, self.B)
        return tf.concat([tf.cos(projected), tf.sin(projected)], axis=-1)

In [ ]:
class MLPLayer(tf.keras.layers.Layer):
    def __init__(self, units, activation='tanh', **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        glorot_init = tf.keras.initializers.GlorotNormal(seed=seed)
        self.dense = tf.keras.layers.Dense(
            self.units,
            dtype=tf.float32,
            kernel_initializer=glorot_init
        )

    def call(self, inputs):
        z = self.dense(inputs)
        return self.activation(z)

In [ ]:
@tf.keras.utils.register_keras_serializable()
class PINN(tf.keras.Model):
    def __init__(self, num_layers=5, hidden_dim=50, L=80.0, T_max=40.0, X_min=-40.0, X_max=40.0, activation='tanh',
                 use_fourier=False, num_fourier_features=64, sigma=1.0, **kwargs):
        super().__init__(**kwargs)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.L = L

        # Fourier Feature Embedding (igual que en M4, mABU y MABUR)
        #   use_fourier = True  -> se activa el embedding: (x,t) -> [cos(2π v B), sin(2π v B)]
        #   use_fourier = False -> la red recibe (x_norm, t_norm) directamente
        self.use_fourier = use_fourier
        self.num_fourier_features = num_fourier_features
        self.sigma = sigma
        self.activation = activation

        # Normalizaciones internas en Float32
        self.T_max = tf.constant(T_max, dtype=tf.float32)
        self.X_min = tf.constant(X_min, dtype=tf.float32)
        self.X_max = tf.constant(X_max, dtype=tf.float32)

    def build(self, input_shape):
        # Opcional: Embedding Fourier
        if self.use_fourier:
            self.fourier = FourierEmbedding(
                num_features=self.num_fourier_features,
                sigma=self.sigma,
                input_dim=2,
                name='fourier_embedding'
            )

        self.hidden_layers = []
        for i in range(self.num_layers):
            self.hidden_layers.append(
                MLPLayer(self.hidden_dim, activation=self.activation, name=f"MLPLayer_{i}")
            )
        self.out_layer = tf.keras.layers.Dense(1, dtype=tf.float32, kernel_initializer='glorot_uniform', name="Output")
        super().build(input_shape)

    def call(self, inputs):
        # Los inputs (x, t) llegan en float64 desde la física, los bajamos a float32 para la red
        x = tf.cast(inputs[0], tf.float32)
        t = tf.cast(inputs[1], tf.float32)

        x_norm = 2.0 * (x - self.X_min) / (self.X_max - self.X_min) - 1.0
        t_norm = t / self.T_max

        h = tf.concat([x_norm, t_norm], axis=-1)

        # Embedding o características directas
        if self.use_fourier:
            h = self.fourier(h)

        for layer in self.hidden_layers:
            h = layer(h)

        N_xt = self.out_layer(h)

        # UPCAST: Devolvemos float64 para que el Tape calcule residuos físicos con alta precisión
        return tf.cast(N_xt, tf.float64)

In [ ]:
def pde_loss_causal(model, x_f, t_f, c_val, m_val, epsilon=100.0, num_chunks=10):
    t_flat = tf.reshape(t_f, [-1])
    sorted_indices = tf.argsort(t_flat)

    t_f_sorted = tf.gather(t_f, sorted_indices)
    x_f_sorted = tf.gather(x_f, sorted_indices)

    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch([x_f_sorted, t_f_sorted])
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch([x_f_sorted, t_f_sorted])
            phi = model([x_f_sorted, t_f_sorted])

        phi_x = tape1.gradient(phi, x_f_sorted)
        phi_t = tape1.gradient(phi, t_f_sorted)

    phi_xx = tape2.gradient(phi_x, x_f_sorted)
    phi_tt = tape2.gradient(phi_t, t_f_sorted)
    del tape1, tape2

    # 3. RESIDUO BRUTO (Element-wise)
    c_tf = tf.cast(c_val, tf.float64)
    m_tf = tf.cast(m_val, tf.float64)

    # Residuo punto por punto (vector columna)
    residual = phi_tt - tf.square(c_tf) * phi_xx + tf.square(m_tf) * phi
    squared_residual = tf.square(residual)

    N_total = tf.shape(squared_residual)[0]
    chunk_size = N_total // num_chunks

    N_exact = chunk_size * num_chunks
    sq_res_trunc = squared_residual[:N_exact]
    chunks = tf.reshape(sq_res_trunc, [num_chunks, chunk_size])

    L_i = tf.reduce_mean(chunks, axis=1)
    cumulative_loss = tf.math.cumsum(L_i, exclusive=True)

    epsilon_tf = tf.constant(epsilon, dtype=tf.float64)
    causal_weights = tf.exp(-epsilon_tf * cumulative_loss)

    causal_weights = tf.stop_gradient(causal_weights)
    causal_loss = tf.reduce_mean(causal_weights * L_i)

    return causal_loss

def initial_condition_loss(model, x_ic, t_ic, phi0_tf, pi0_tf):
    with tf.GradientTape() as tape:
        tape.watch(t_ic)
        phi_pred = model([x_ic, t_ic])
    phi_t_pred = tape.gradient(phi_pred, t_ic)

    loss_phi = tf.reduce_mean(tf.square(phi_pred - phi0_tf))
    loss_pi  = tf.reduce_mean(tf.square(phi_t_pred - pi0_tf))
    return loss_phi, loss_pi

def boundary_condition_loss(model, t_bc, x_bc_left, x_bc_right):
    with tf.GradientTape() as tape_l:
        tape_l.watch(x_bc_left)
        phi_left = model([x_bc_left, t_bc])
    phi_x_left = tape_l.gradient(phi_left, x_bc_left)

    with tf.GradientTape() as tape_r:
        tape_r.watch(x_bc_right)
        phi_right = model([x_bc_right, t_bc])
    phi_x_right = tape_r.gradient(phi_right, x_bc_right)

    loss_val = tf.reduce_mean(tf.square(phi_left - phi_right))
    loss_der = tf.reduce_mean(tf.square(phi_x_left - phi_x_right))

    return loss_val + loss_der

In [ ]:
@tf.function(jit_compile=True ,reduce_retracing=True)
def train_step(model, optimizer, x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc, c_tf, m_tf, alpha_lra,
               c_phi=0.0, c_pi=0.0, c_bc = 0.0, w_tensor=[1.0 ,1.0 ,1.0 ,1.0], epsilon = 1.0, num_chunks = 16):
    with tf.GradientTape(persistent=True) as tape:
        lf = pde_loss_causal(model, x_f, t_f, c_tf, m_tf, epsilon= epsilon, num_chunks= num_chunks)
        l_phi, l_pi = initial_condition_loss(model, x_ic, t_ic, phi0, pi0)
        lb = boundary_condition_loss(model, t_bc, x_bc_l, x_bc_r)

        total_loss = w_tensor[0]*lf + w_tensor[1]*l_phi + w_tensor[2]*l_pi + w_tensor[3]*lb
        tf.debugging.check_numerics(total_loss, "total_loss NaN/Inf")

    #Algoritmo de LRA
    target_layer_vars = model.trainable_variables[-2]

    grad_f = tape.gradient(lf, target_layer_vars)
    grad_phi = tape.gradient(l_phi, target_layer_vars)
    grad_pi = tape.gradient(l_pi, target_layer_vars)
    grad_b = tape.gradient(lb, target_layer_vars)

    max_f = tf.cast(tf.reduce_max(tf.abs(grad_f)), tf.float64)
    mean_phi = tf.cast(tf.reduce_mean(tf.abs(grad_phi)), tf.float64)
    mean_pi = tf.cast(tf.reduce_mean(tf.abs(grad_pi)), tf.float64)
    mean_b = tf.cast(tf.reduce_mean(tf.abs(grad_b)), tf.float64)

    eps = tf.constant(np.finfo(float).eps, dtype=tf.float64)

    #Por si las moscas
    den_phi = mean_phi + c_phi*max_f
    den_pi = mean_pi + c_pi*max_f
    den_bc = mean_b + c_bc-max_f

    # Calculamos los multiplicadores crudos
    lamb_phi = max_f / (den_phi + eps)
    lamb_pi = max_f / (den_pi + eps)
    lamb_b = max_f / (den_bc + eps)

    lamb_phi = tf.clip_by_value(lamb_phi, 1e-1, 1e+2)
    lamb_pi  = tf.clip_by_value(lamb_pi,  1e-2, 1e+1)
    lamb_b  = tf.clip_by_value(lamb_b,  1e-3, 1e-1)

    # Actualizamos los pesos mediante Media Móvil (EMA) directamente en la GPU
    w_phi_new = (1.0 - alpha_lra) * w_tensor[1] + alpha_lra * lamb_phi
    w_pi_new = (1.0 - alpha_lra) * w_tensor[2] + alpha_lra * lamb_pi
    w_bc_new = (1.0 - alpha_lra) * w_tensor[3] + alpha_lra * lamb_b

    # Asignamos los nuevos pesos a la variable global (Entrarán en efecto en la próxima época)
    w_tensor.assign([w_tensor[0], w_phi_new, w_pi_new, w_bc_new])

    # OPTIMIZACIÓN
    grads = tape.gradient(total_loss, model.trainable_variables)
    del tape

    grads = [tf.zeros_like(v) if g is None else g for g, v in zip(grads, model.trainable_variables)]
    grads, gnorm = tf.clip_by_global_norm(grads, 50.0)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    return total_loss, lf, l_phi, l_pi, lb, gnorm

def resample_all_focused(n_phys, n_bc, n_ic, focus=True):
    """ Muestreo focalizado: cono+global para PDE, gaussiano+uniforme para IC """

    # ============ PDE: 80% cono de luz + 20% global ============
    if focus:
        n_cone = int(0.8 * n_phys)
        n_global = n_phys - n_cone

        # Cono (alta densidad donde hay acción)
        t_cone = np.random.uniform(0.0, T_max, (n_cone, 1)).astype(np.float64)
        margen = 5.0
        radios = sigma + c * t_cone + margen
        radios = np.clip(radios, 0.0, X_max)
        u = np.random.uniform(-1.0, 1.0, (n_cone, 1)).astype(np.float64)
        x_cone = u * radios

        # Global (enseña φ≈0 fuera del cono)
        t_global = np.random.uniform(0.0, T_max, (n_global, 1)).astype(np.float64)
        x_global = np.random.uniform(X_min, X_max, (n_global, 1)).astype(np.float64)

        x_f_new = np.concatenate([x_cone, x_global], axis=0)
        t_f_new = np.concatenate([t_cone, t_global], axis=0)
    else:
        t_f_new = np.random.uniform(0.0, T_max, (n_phys, 1)).astype(np.float64)
        x_f_new = np.random.uniform(X_min, X_max, (n_phys, 1)).astype(np.float64)

    # ============ BC: fronteras ============
    t_bc_new = np.random.uniform(0.0, T_max, (n_bc, 1)).astype(np.float64)
    x_bc_l_new = np.full((n_bc, 1), X_min, dtype=np.float64)
    x_bc_r_new = np.full((n_bc, 1), X_max, dtype=np.float64)

    # ============ IC: 80% gaussiano + 20% uniforme ============
    if focus:
        n_ic_focus = int(0.8 * n_ic)
        n_ic_unif  = n_ic - n_ic_focus
        x_ic_focus = np.clip(np.random.normal(0.0, 4.0 * sigma, (n_ic_focus, 1)),
                             X_min, X_max)
        x_ic_unif  = np.random.uniform(X_min, X_max, (n_ic_unif, 1))
        x_ic_new   = np.concatenate([x_ic_focus, x_ic_unif], axis=0).astype(np.float64)
    else:
        x_ic_new = np.random.uniform(X_min, X_max, (n_ic, 1)).astype(np.float64)

    t_ic_new = np.zeros((n_ic, 1), dtype=np.float64)
    phi0_new = interp_phi0(x_ic_new.flatten()).reshape(-1, 1).astype(np.float64)
    pi0_new  = interp_pi0(x_ic_new.flatten()).reshape(-1, 1).astype(np.float64)

    return (tf.constant(x_f_new), tf.constant(t_f_new),
            tf.constant(x_ic_new), tf.constant(t_ic_new),
            tf.constant(phi0_new), tf.constant(pi0_new),
            tf.constant(x_bc_l_new), tf.constant(x_bc_r_new), tf.constant(t_bc_new))

In [ ]:
@tf.function(reduce_retracing=True)
def get_loss_and_grads_lbfgs(x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc, c_tf, m_tf, epsilon, num_chunks):
    with tf.GradientTape() as tape:
        lf = pde_loss_causal(model, x_f, t_f, c_tf, m_tf, epsilon = epsilon, num_chunks= num_chunks)
        l_phi, l_pi = initial_condition_loss(model, x_ic, t_ic, phi0, pi0)
        lb = boundary_condition_loss(model, t_bc, x_bc_l, x_bc_r)
        total_loss = lf + l_phi + l_pi + lb

    grads = tape.gradient(total_loss, model.trainable_variables)

    # Aplanamos los gradientes y los escalamos a Float64 para L-BFGS (Scipy lo requiere)
    flat_grads = tf.cast(tf.concat([tf.reshape(g, [-1]) for g in grads if g is not None], axis=0), tf.float64)

    return total_loss, lf, l_phi, l_pi, lb, flat_grads


def set_flat_weights(model, flat_weights):
    flat_weights = tf.cast(flat_weights, tf.float32)
    start = 0
    for w in model.trainable_variables:
        shape = w.shape
        size = tf.reduce_prod(shape)
        new_w = tf.reshape(flat_weights[start:start + size], shape)
        w.assign(new_w)
        start += size


def get_flat_weights(model):
    weights = [tf.reshape(tf.cast(w, tf.float64), [-1]) for w in model.trainable_variables]
    return tf.concat(weights, axis=0).numpy()


def lbfgs_objective(flat_weights, x_f, t_f, x_ic, t_ic, phi0, pi0,
                    x_bc_l, x_bc_r, t_bc, c_tf, m_tf, epsilon, num_chunks):
    set_flat_weights(model, flat_weights)
    t_loss, lf, l_phi, l_pi, lb, grads = get_loss_and_grads_lbfgs(
        x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc,
        c_tf, m_tf, epsilon, num_chunks
    )

    loss_val = t_loss.numpy().astype(np.float64)
    grads_val = grads.numpy().astype(np.float64)

    global lbfgs_iter
    lbfgs_iter += 1

    if lbfgs_iter % 100 == 0:
        print(f"L-BFGS Iter {lbfgs_iter} | Total Loss={loss_val:.2e} | "
              f"PDE={lf:.2e} | PHI={l_phi:.2e} | PI={l_pi:.2e} | BC={lb:.2e}")
        loss_history["epoch"].append(epochs_adam + lbfgs_iter)
        loss_history["total"].append(loss_val)
        loss_history["pde"].append(lf.numpy())
        loss_history["phi"].append(l_phi.numpy())
        loss_history["pi"].append(l_pi.numpy())
        loss_history["bc"].append(lb.numpy())

    return loss_val, grads_val

## **2b. ANaGRAM (gradiente natural empírico)**

Funciones nuevas (no reemplazan nada de Adam). Port a TensorFlow del Algoritmo 2 de Schwencke & Furtlehner (ICLR 2025) y de `nat_grad_factory` / `grid_line_search_factory` del repo `Natural-Gradient-PINNs-ICML23-ANaGRAM`.

In [ ]:
# =====================================================================
#  ANaGRAM — "A Natural Gradient Relative to Adapted Model"
#  Schwencke & Furtlehner, ICLR 2025 (arXiv:2412.10782), Algoritmo 2.
#
#  Port a TensorFlow de las funciones del repo (JAX) de Müller & Zeinhofer
#  en Código/Natural-Gradient-PINNs-ICML23-ANaGRAM:
#     ngrad/gram.py    -> gram_factory / nat_grad_factory  => anagram_direccion(metodo="engd")
#     ngrad/utility.py -> grid_line_search_factory         => anagram_line_search
#  y la dirección propia de ANaGRAM (SVD truncada del Jacobiano de residuos):
#     J = U Σ Vᵀ ,  d = V Σ⁺ Uᵀ r ,  θ ← θ − η d   (η por line search)
#
#  Todo es NUEVO: no modifica pde_loss_causal, train_step ni el modelo.
#  Funciona con cualquier modelo que se llame como model([x, t]) (PINN, M4,
#  mABU, MABUR), así que estas celdas se pueden copiar a los otros notebooks.
# =====================================================================

# ---------- 1. Residuos punto a punto (copias de las pérdidas originales) ----------

def residuo_pde_kg(model, x, t, c_val, m_val):
    """Residuo de Klein-Gordon φ_tt − c²φ_xx + m²φ (copia de pde_loss_causal, sin promediar)."""
    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch([x, t])
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch([x, t])
            phi = model([x, t])
        phi_x = tape1.gradient(phi, x)
        phi_t = tape1.gradient(phi, t)
    phi_xx = tape2.gradient(phi_x, x)
    phi_tt = tape2.gradient(phi_t, t)
    del tape1, tape2

    c_tf = tf.cast(c_val, tf.float64)
    m_tf = tf.cast(m_val, tf.float64)
    return phi_tt - tf.square(c_tf) * phi_xx + tf.square(m_tf) * phi


def residuo_ic_phi_kg(model, x_ic, t_ic, phi0):
    """φ(x,0) − φ0(x)  (copia de initial_condition_loss)."""
    return model([x_ic, t_ic]) - phi0


def residuo_ic_pi_kg(model, x_ic, t_ic, pi0):
    """φ_t(x,0) − π0(x)  (copia de initial_condition_loss)."""
    with tf.GradientTape() as tape:
        tape.watch(t_ic)
        phi_pred = model([x_ic, t_ic])
    # "+ 0·φ" no cambia el valor: mantiene conectados todos los parámetros (p. ej. el bias de salida,
    # que no afecta φ_t). Sin esto TF falla al derivar respecto a θ dentro de tf.vectorized_map.
    return tape.gradient(phi_pred, t_ic) - pi0 + 0.0 * phi_pred


def residuo_bc_val_kg(model, t_bc, x_bc_left, x_bc_right):
    """φ(X_min,t) − φ(X_max,t)  (copia de boundary_condition_loss)."""
    return model([x_bc_left, t_bc]) - model([x_bc_right, t_bc])


def residuo_bc_der_kg(model, t_bc, x_bc_left, x_bc_right):
    """φ_x(X_min,t) − φ_x(X_max,t)  (copia de boundary_condition_loss)."""
    with tf.GradientTape() as tape_l:
        tape_l.watch(x_bc_left)
        phi_left = model([x_bc_left, t_bc])
    phi_x_left = tape_l.gradient(phi_left, x_bc_left)
    with tf.GradientTape() as tape_r:
        tape_r.watch(x_bc_right)
        phi_right = model([x_bc_right, t_bc])
    phi_x_right = tape_r.gradient(phi_right, x_bc_right)
    return phi_x_left - phi_x_right + 0.0 * (phi_left - phi_right)   # "+ 0·φ": ver residuo_ic_pi_kg


# ---------- 2. Pesos causales por punto (misma regla que pde_loss_causal) ----------

@tf.function(reduce_retracing=True)
def anagram_pesos_causales(model, x_f, t_f, c_val, m_val, epsilon, num_chunks):
    """Ordena los puntos PDE por t y devuelve el coeficiente de cada punto tal que
       Σ_j coef_j · r_j²  ==  pde_loss_causal(...)   (pesos causales congelados, como stop_gradient)."""
    sorted_indices = tf.argsort(tf.reshape(t_f, [-1]))
    t_s = tf.gather(t_f, sorted_indices)
    x_s = tf.gather(x_f, sorted_indices)

    res = residuo_pde_kg(model, x_s, t_s, c_val, m_val)
    sq = tf.reshape(tf.square(res), [-1])

    n_total = tf.shape(sq)[0]
    chunk_size = n_total // num_chunks
    n_exact = chunk_size * num_chunks
    L_i = tf.reduce_mean(tf.reshape(sq[:n_exact], [num_chunks, chunk_size]), axis=1)
    cumulative_loss = tf.math.cumsum(L_i, exclusive=True)
    w_i = tf.exp(-tf.cast(epsilon, tf.float64) * cumulative_loss)

    w_pt = tf.repeat(w_i, chunk_size)
    w_pt = tf.concat([w_pt, tf.zeros([n_total - n_exact], tf.float64)], axis=0)   # puntos truncados -> peso 0
    coef = w_pt / tf.cast(n_exact, tf.float64)
    return x_s, t_s, tf.reshape(coef, [-1, 1])


# ---------- 2b. Frente causal y ε adaptativo (variante propia, no está en los papers) ----------

@tf.function(reduce_retracing=True)
def anagram_pesos_causales_frente(model, x_f, t_f, c_val, m_val, epsilon, num_chunks):
    """Igual que anagram_pesos_causales, pero devuelve también los pesos por tramo w_i
       (para seguir el "frente causal"). epsilon puede ser un tensor (evita re-trazar al cambiarlo)."""
    sorted_indices = tf.argsort(tf.reshape(t_f, [-1]))
    t_s = tf.gather(t_f, sorted_indices)
    x_s = tf.gather(x_f, sorted_indices)

    res = residuo_pde_kg(model, x_s, t_s, c_val, m_val)
    sq = tf.reshape(tf.square(res), [-1])

    n_total = tf.shape(sq)[0]
    chunk_size = n_total // num_chunks
    n_exact = chunk_size * num_chunks
    L_i = tf.reduce_mean(tf.reshape(sq[:n_exact], [num_chunks, chunk_size]), axis=1)
    cumulative_loss = tf.math.cumsum(L_i, exclusive=True)
    w_i = tf.exp(-tf.cast(epsilon, tf.float64) * cumulative_loss)

    w_pt = tf.repeat(w_i, chunk_size)
    w_pt = tf.concat([w_pt, tf.zeros([n_total - n_exact], tf.float64)], axis=0)
    coef = w_pt / tf.cast(n_exact, tf.float64)
    return x_s, t_s, tf.reshape(coef, [-1, 1]), w_i


class ControlCausal:
    """
    Sigue el frente causal y (opcional) adapta ε:
      - tramos abiertos = #{tramos con peso causal w_i > umbral_tramo}
      - adaptativo: si el récord de tramos abiertos no mejora en `paciencia` iteraciones y el último
        tramo aún no está abierto, ε <- max(ε·factor, eps_min). (Wang et al. SUBEN ε por etapas con Adam;
        con Gauss-Newton conviene empezar alto y bajarlo para no trabar el frente.)
      - dominio abierto: peso del último tramo > umbral_abierto durante n_confirmar iteraciones seguidas.
    """
    def __init__(self, eps_inicial, adaptativo=False, eps_min=10.0, paciencia=20, factor=0.5,
                 umbral_tramo=0.5, umbral_abierto=0.99, n_confirmar=5):
        self.eps = float(eps_inicial)
        self.adaptativo, self.eps_min, self.paciencia, self.factor = adaptativo, float(eps_min), int(paciencia), factor
        self.umbral_tramo, self.umbral_abierto, self.n_confirmar = umbral_tramo, umbral_abierto, n_confirmar
        self.record, self.sin_avance, self.racha_abierto, self.it_abierto = -1, 0, 0, None

    def actualizar(self, w_i, it):
        w = np.asarray(w_i, dtype=np.float64)
        n_ab, w_ult = int(np.sum(w > self.umbral_tramo)), float(w[-1])
        if n_ab > self.record:
            self.record, self.sin_avance = n_ab, 0
        else:
            self.sin_avance += 1
        cambio = False
        if (self.adaptativo and self.sin_avance >= self.paciencia and w_ult < self.umbral_abierto
                and self.eps > self.eps_min):
            self.eps = max(self.eps * self.factor, self.eps_min)
            self.record, self.sin_avance, cambio = -1, 0, True      # se vuelve a medir el frente con el nuevo ε
        self.racha_abierto = self.racha_abierto + 1 if w_ult > self.umbral_abierto else 0
        if self.it_abierto is None and self.racha_abierto >= self.n_confirmar:
            self.it_abierto = it
        return n_ab, w_ult, cambio


def anagram_escalas(coef_pde, w, n_ic, n_bc):
    """Escala de cada fila: r_k = sqrt(w_k · coef_k) · residuo_k, de modo que ‖r‖² = pérdida total."""
    w = tf.cast(w, tf.float64)
    s_pde = tf.sqrt(w[0] * coef_pde)
    s_phi = tf.sqrt(w[1] / n_ic)
    s_pi  = tf.sqrt(w[2] / n_ic)
    s_bc  = tf.sqrt(w[3] / n_bc)
    return s_pde, s_phi, s_pi, s_bc


# ---------- 3. Vector de residuos r (por lote) ----------

@tf.function(reduce_retracing=True)
def anagram_vector_residuos(model, x_s, t_s, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc,
                            coef_pde, w, c_val, m_val):
    n_ic = tf.cast(tf.shape(x_ic)[0], tf.float64)
    n_bc = tf.cast(tf.shape(t_bc)[0], tf.float64)
    s_pde, s_phi, s_pi, s_bc = anagram_escalas(coef_pde, w, n_ic, n_bc)

    r_pde = residuo_pde_kg(model, x_s, t_s, c_val, m_val)
    r_phi = residuo_ic_phi_kg(model, x_ic, t_ic, phi0)
    r_pi  = residuo_ic_pi_kg(model, x_ic, t_ic, pi0)
    r_bv  = residuo_bc_val_kg(model, t_bc, x_bc_l, x_bc_r)
    r_bd  = residuo_bc_der_kg(model, t_bc, x_bc_l, x_bc_r)

    r = tf.concat([s_pde * r_pde, s_phi * r_phi, s_pi * r_pi, s_bc * r_bv, s_bc * r_bd], axis=0)
    r = tf.reshape(r, [-1])

    # Pérdidas por término (mismas definiciones que en el entrenamiento con Adam)
    lf    = tf.reduce_sum(coef_pde * tf.square(r_pde))
    l_phi = tf.reduce_mean(tf.square(r_phi))
    l_pi  = tf.reduce_mean(tf.square(r_pi))
    lb    = tf.reduce_mean(tf.square(r_bv)) + tf.reduce_mean(tf.square(r_bd))
    total = tf.reduce_sum(tf.square(r))      # = w0·lf + w1·l_phi + w2·l_pi + w3·lb
    return r, total, lf, l_phi, l_pi, lb


# ---------- 4. Jacobiano de residuos  φ̂_θ = (∂_p r_i)  (línea 2 del Algoritmo 2) ----------

def _crear_jac_por_punto(fn_residuo, n_extra):
    """Crea una tf.function que calcula, punto a punto (tf.vectorized_map), la fila ∂_θ r_i.
       Hacerlo por punto evita el costo O(S²) en memoria de tape.jacobian sobre un lote."""
    @tf.function(reduce_retracing=True)
    def jac_chunk(model, *elems_y_consts):
        elems = elems_y_consts[:len(elems_y_consts) - n_extra]
        consts = elems_y_consts[len(elems_y_consts) - n_extra:]
        variables = model.trainable_variables

        def por_punto(e):
            e = [tf.reshape(v, [1, 1]) for v in e]
            with tf.GradientTape() as tp:
                r_i = tf.reshape(fn_residuo(model, *e, *consts), [])
            g = tp.gradient(r_i, variables, unconnected_gradients=tf.UnconnectedGradients.ZERO)
            return tf.concat([tf.reshape(tf.cast(gi, tf.float64), [-1]) for gi in g], axis=0)

        return tf.vectorized_map(por_punto, list(elems))
    return jac_chunk


_jac_pde    = _crear_jac_por_punto(lambda mdl, x, t, c_v, m_v: residuo_pde_kg(mdl, x, t, c_v, m_v), n_extra=2)
_jac_ic_phi = _crear_jac_por_punto(lambda mdl, x, t, p0: residuo_ic_phi_kg(mdl, x, t, p0), n_extra=0)
_jac_ic_pi  = _crear_jac_por_punto(lambda mdl, x, t, q0: residuo_ic_pi_kg(mdl, x, t, q0), n_extra=0)
_jac_bc_val = _crear_jac_por_punto(lambda mdl, t, xl, xr: residuo_bc_val_kg(mdl, t, xl, xr), n_extra=0)
_jac_bc_der = _crear_jac_por_punto(lambda mdl, t, xl, xr: residuo_bc_der_kg(mdl, t, xl, xr), n_extra=0)


def _jac_en_chunks(jac_fn, model, elems, consts, tam_chunk):
    """Aplica jac_fn por bloques de tamaño fijo (rellena el último para no re-trazar)."""
    n = int(elems[0].shape[0])
    filas = []
    for i in range(0, n, tam_chunk):
        trozo = [e[i:i + tam_chunk] for e in elems]
        m_real = int(trozo[0].shape[0])
        if m_real < tam_chunk:   # relleno repitiendo el último punto
            trozo = [tf.concat([e, tf.repeat(e[-1:], tam_chunk - m_real, axis=0)], axis=0) for e in trozo]
        filas.append(jac_fn(model, *trozo, *consts)[:m_real])
    return tf.concat(filas, axis=0)


def anagram_jacobiano(model, x_s, t_s, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc,
                      coef_pde, w, c_val, m_val, tam_chunk=256):
    """Devuelve J (S × P, float64), en el mismo orden de filas que anagram_vector_residuos."""
    n_ic = tf.cast(tf.shape(x_ic)[0], tf.float64)
    n_bc = tf.cast(tf.shape(t_bc)[0], tf.float64)
    s_pde, s_phi, s_pi, s_bc = anagram_escalas(coef_pde, w, n_ic, n_bc)
    c64, m64 = tf.cast(c_val, tf.float64), tf.cast(m_val, tf.float64)

    J_pde = _jac_en_chunks(_jac_pde,    model, [x_s, t_s],                 [c64, m64], tam_chunk)
    J_phi = _jac_en_chunks(_jac_ic_phi, model, [x_ic, t_ic, phi0],         [], tam_chunk)
    J_pi  = _jac_en_chunks(_jac_ic_pi,  model, [x_ic, t_ic, pi0],          [], tam_chunk)
    J_bv  = _jac_en_chunks(_jac_bc_val, model, [t_bc, x_bc_l, x_bc_r],     [], tam_chunk)
    J_bd  = _jac_en_chunks(_jac_bc_der, model, [t_bc, x_bc_l, x_bc_r],     [], tam_chunk)

    return tf.concat([s_pde * J_pde, s_phi * J_phi, s_pi * J_pi, s_bc * J_bv, s_bc * J_bd], axis=0)


# ---------- 5. Dirección natural  d = J⁺ r  (líneas 3-6 del Algoritmo 2) ----------

def anagram_direccion(J, r, rcond=1e-6, metodo="auto", precision="float64"):
    """
    metodo = "svd"  : Algoritmo 2 literal. SVD de J (S×P), corta σ ≤ rcond·σ_max, d = V Σ⁺ Uᵀ r.
             "ntk"  : mismo d vía el núcleo empírico K = J Jᵀ (S×S) = U Σ² Uᵀ:
                      d = Jᵀ U Σ⁻² Uᵀ r. Costo O(S²P): es lo que da la complejidad min(P²S, S²P).
             "engd" : port de nat_grad_factory (ENGD, Müller & Zeinhofer): lstsq(G, ∇L) con
                      G = JᵀJ (P×P) y ∇L = Jᵀr. Mismo d, pero O(P³): solo para redes pequeñas.
             "auto" : "ntk" si S ≤ P, si no "svd".
    precision = "float64" (exacto) | "float32" (≈10-30x más rápido en GPUs T4/L4, que son lentas en float64;
                válido si rcond ≥ ~1e-3, porque float32 no resuelve σ < ~3e-4·σ_max en K = J Jᵀ).
    Devuelve d (P,), rango efectivo y σ_max.
    """
    S, P = int(J.shape[0]), int(J.shape[1])
    if metodo == "auto":
        metodo = "ntk" if S <= P else "svd"
    dt = tf.float32 if precision == "float32" else tf.float64
    J = tf.cast(J, dt)
    r = tf.reshape(tf.cast(r, dt), [-1, 1])

    if metodo == "svd":
        s, U, V = tf.linalg.svd(J, full_matrices=False)
        s_max = s[0]
        keep = s > rcond * s_max
        s_inv = tf.where(keep, 1.0 / tf.where(keep, s, tf.ones_like(s)), tf.zeros_like(s))
        d = V @ (s_inv[:, None] * (tf.transpose(U) @ r))
        rango = tf.reduce_sum(tf.cast(keep, tf.int32))

    elif metodo == "ntk":
        K = J @ tf.transpose(J)
        lam, Q = tf.linalg.eigh(K)                 # λ = σ² (orden ascendente)
        lam_max = lam[-1]
        keep = lam > (rcond ** 2) * lam_max         # σ > rcond·σ_max  <=>  λ > rcond²·λ_max
        lam_inv = tf.where(keep, 1.0 / tf.where(keep, lam, tf.ones_like(lam)), tf.zeros_like(lam))
        d = tf.transpose(J) @ (Q @ (lam_inv[:, None] * (tf.transpose(Q) @ r)))
        s_max = tf.sqrt(lam_max)
        rango = tf.reduce_sum(tf.cast(keep, tf.int32))

    elif metodo == "engd":
        # nat_grad_factory: lstsq(G, ∇L). jnp.linalg.lstsq corta σ(G) ≤ eps·P·σ_max(G); aquí igual vía eigh.
        G = tf.transpose(J) @ J
        grad = tf.transpose(J) @ r
        lam, Q = tf.linalg.eigh(G)
        lam_max = lam[-1]
        keep = lam > np.finfo(dt.as_numpy_dtype).eps * P * lam_max
        lam_inv = tf.where(keep, 1.0 / tf.where(keep, lam, tf.ones_like(lam)), tf.zeros_like(lam))
        d = Q @ (lam_inv[:, None] * (tf.transpose(Q) @ grad))
        s_max = tf.sqrt(lam_max)
        rango = tf.reduce_sum(tf.cast(keep, tf.int32))
    else:
        raise ValueError(f"metodo desconocido: {metodo}")

    d = tf.cast(tf.reshape(d, [-1]), tf.float64)
    # Igual que nat_grad_factory: si sale NaN, dirección cero
    if not bool(tf.reduce_all(tf.math.is_finite(d))):
        d = tf.zeros_like(d)
    return d, int(rango), float(s_max)


# ---------- 6. Line search en rejilla (port de grid_line_search_factory) ----------

def _desaplanar(model, flat):
    partes, inicio = [], 0
    for v in model.trainable_variables:
        n = int(np.prod(v.shape))
        partes.append(tf.cast(tf.reshape(flat[inicio:inicio + n], v.shape), v.dtype))
        inicio += n
    return partes


def anagram_line_search(model, d_flat, pasos, lote, coef_pde, w, c_val, m_val):
    """η = argmin_{η ∈ pasos} ‖r(θ − η d)‖²  sobre el mismo lote (línea 7 del Algoritmo 2)."""
    variables = model.trainable_variables
    theta0 = [tf.identity(v) for v in variables]
    d_list = _desaplanar(model, d_flat)

    perdidas = []
    for eta in pasos:
        for v, th, dv in zip(variables, theta0, d_list):
            v.assign(th - tf.cast(eta, v.dtype) * dv)
        _, total, *_ = anagram_vector_residuos(model, *lote, coef_pde, w, c_val, m_val)
        perdidas.append(float(total))

    perdidas = np.nan_to_num(np.array(perdidas), nan=np.inf)
    k = int(np.argmin(perdidas))
    eta_opt = float(pasos[k])
    for v, th, dv in zip(variables, theta0, d_list):
        v.assign(th - tf.cast(eta_opt, v.dtype) * dv)
    return eta_opt, float(perdidas[k])


# ---------- 7. Bucle de entrenamiento ANaGRAM ----------

def _guardar_historial_anagram(loss_history, ruta_csv):
    valid = {k: v for k, v in loss_history.items() if len(v) > 0}
    n = min(len(v) for v in valid.values())
    pd.DataFrame({k: v[:n] for k, v in valid.items()}).to_csv(ruta_csv, index=False)


def entrenar_anagram(model, loss_history, iteraciones, n_phys, n_bc, n_ic, c_val, m_val,
                     w=(1.0, 1.0, 1.0, 1.0), epsilon=10.0, num_chunks=16,
                     rcond=1e-6, metodo="auto", tam_chunk=256,
                     pasos=0.5 ** np.linspace(0, 30, 31),
                     as_freq=1, stop_focus=None, print_every=50, precision="float64",
                     frac_stop_resampling=1.0, epsilon_adaptativo=False, epsilon_min=10.0,
                     paciencia_causal=20, iter_extra_tras_abrir=None,
                     ruta_csv="training_loss_history_ANaGRAM.csv",
                     ruta_pesos="ANaGRAM_weights.weights.h5"):
    """
    ANaGRAM para la PINN de Klein-Gordon. Reutiliza resample_all_focused (mismo muestreo que Adam)
    y la pérdida causal (pesos congelados en cada paso). Sin LRA: los pesos w quedan fijos.
    - El historial se registra en CADA iteración y se guarda en disco (CSV + pesos) cada print_every.
    - Si se interrumpe la celda (botón stop / KeyboardInterrupt) se guarda lo que haya y la celda
      de entrenamiento continúa normalmente con su guardado final.
    - Causal: `epsilon` es el ε inicial. epsilon_adaptativo=True lo baja a la mitad (hasta epsilon_min)
      si el frente causal no avanza en paciencia_causal iteraciones (ControlCausal). Se registra el frente
      (tramos abiertos y peso del último tramo). iter_extra_tras_abrir=N detiene N iteraciones después de
      abrir todo el dominio (w_último > 0.99). frac_stop_resampling: deja de remuestrear desde esa fracción.
    Devuelve el último lote (por si se quiere seguir con L-BFGS).
    """
    w = tf.constant(w, dtype=tf.float64)
    if stop_focus is None:
        stop_focus = 0.85 * iteraciones
    stop_resampling = frac_stop_resampling * iteraciones     # desde aquí el lote queda fijo (1.0 = nunca)
    ctrl = ControlCausal(epsilon, adaptativo=epsilon_adaptativo, eps_min=epsilon_min, paciencia=paciencia_causal)
    for k in ["eps_causal", "tramos_abiertos", "w_ultimo"]:
        loss_history.setdefault(k, [])
    for k in ["eta", "rango", "s_max", "t_jac", "t_dir", "t_ls"]:
        loss_history.setdefault(k, [])
    if metodo == "svd" and tf.config.list_physical_devices("GPU"):
        print("AVISO: tf.linalg.svd es muy lento en GPU. Con S < P usa metodo='ntk' o 'auto' (mismo resultado).")

    start_time = time.time()
    try:
        for it in range(1, iteraciones + 1):
            if it == 1 or (as_freq > 0 and it % as_freq == 0 and it <= stop_resampling):
                current_focus = it <= stop_focus
                x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = resample_all_focused(
                    n_phys, n_bc, n_ic, focus=current_focus)

            # (a) pesos causales congelados en θ_t
            eps_usado = ctrl.eps
            x_s, t_s, coef_pde, w_tramos = anagram_pesos_causales_frente(
                model, x_f, t_f, c_val, m_val, tf.constant(eps_usado, tf.float64), num_chunks)
            n_abiertos, w_ult, cambio_eps = ctrl.actualizar(w_tramos.numpy(), it)
            if cambio_eps:
                print(f"   ε causal -> {ctrl.eps:g} (el frente no avanzó en {paciencia_causal} iteraciones)")
            if ctrl.it_abierto == it:
                print(f"   >>> Dominio causal abierto en la iteración {it} (w_último > 0.99)")
            lote = (x_s, t_s, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc)

            # (b) residuos y Jacobiano
            t0 = time.time()
            r, t_loss, lf, l_phi, l_pi, lb = anagram_vector_residuos(model, *lote, coef_pde, w, c_val, m_val)
            J = anagram_jacobiano(model, *lote, coef_pde, w, c_val, m_val, tam_chunk=tam_chunk)
            gnorm = float(2.0 * tf.norm(tf.linalg.matvec(J, r, transpose_a=True)))   # ‖∇_θ total‖

            # (c) dirección natural y (d) line search
            t1 = time.time()
            d, rango, s_max = anagram_direccion(J, r, rcond=rcond, metodo=metodo, precision=precision)
            del J
            t2 = time.time()
            eta, nueva_perdida = anagram_line_search(model, d, pasos, lote, coef_pde, w, c_val, m_val)
            t3 = time.time()
            elapsed = time.time() - start_time

            # Historial en CADA iteración
            loss_history["epoch"].append(it)
            loss_history["total"].append(t_loss.numpy())
            loss_history["pde"].append(lf.numpy())
            loss_history["phi"].append(l_phi.numpy())
            loss_history["pi"].append(l_pi.numpy())
            loss_history["bc"].append(lb.numpy())
            loss_history["lambda_phi"].append(float(w[1]))
            loss_history["lambda_pi"].append(float(w[2]))
            loss_history["lambda_bc"].append(float(w[3]))
            loss_history["gnorm"].append(gnorm)
            loss_history["time"].append(round(elapsed, 1))
            loss_history["eta"].append(eta)
            loss_history["rango"].append(rango)
            loss_history["s_max"].append(s_max)
            loss_history["t_jac"].append(round(t1 - t0, 3))
            loss_history["t_dir"].append(round(t2 - t1, 3))
            loss_history["t_ls"].append(round(t3 - t2, 3))
            loss_history["eps_causal"].append(eps_usado)
            loss_history["tramos_abiertos"].append(n_abiertos)
            loss_history["w_ultimo"].append(w_ult)

            if it % print_every == 0 or it <= 3:
                print(f"ANaGRAM It {it}/{iteraciones} | Total={t_loss:.2e} | PDE={lf:.2e} | PHI={l_phi:.2e} | "
                      f"PI = {l_pi:.2e} | BC={lb:.2e} | eta={eta:.1e} | rango={rango}/{int(r.shape[0])} | "
                      f"gnorm ={gnorm:.2e} | paso: J {t1 - t0:.2f}s + dir {t2 - t1:.2f}s + LS {t3 - t2:.2f}s | "
                      f"ε={eps_usado:g} frente {n_abiertos}/{num_chunks} (w_últ={w_ult:.2f}) | Time: {elapsed:.1f}s")
                _guardar_historial_anagram(loss_history, ruta_csv)
                model.save_weights(ruta_pesos)

            if (iter_extra_tras_abrir is not None and ctrl.it_abierto is not None
                    and it - ctrl.it_abierto >= iter_extra_tras_abrir):
                print(f">>> Detenido: {iter_extra_tras_abrir} iteraciones después de abrir el dominio causal.")
                break

    except KeyboardInterrupt:
        print(f"\n>>> ANaGRAM interrumpido en la iteración {it}. Guardando historial y pesos...")

    _guardar_historial_anagram(loss_history, ruta_csv)
    model.save_weights(ruta_pesos)
    print(f"Historial ANaGRAM -> {ruta_csv} | pesos -> {ruta_pesos}")
    return x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc


## **2c. AMStraMGRAM (corte adaptativo para ANaGRAM)**

Schwencke, Rousselot, Shilova & Furtlehner (arXiv:2510.15998), Algoritmo 2. Usa las funciones de la celda 2b; no modifica nada de Adam ni de ANaGRAM.

In [ ]:
# =====================================================================
#  AMStraMGRAM — "Adaptive Multi-cutoff Strategy Modification for ANaGRAM"
#  Schwencke, Rousselot, Shilova & Furtlehner (arXiv:2510.15998), Algoritmo 2.
#
#  Igual que ANaGRAM, pero el número de componentes que se retienen en la
#  pseudo-inversa (rango de corte) se elige en cada iteración comparando:
#     σ_j    : valores singulares de J (orden decreciente)
#     RCE_j  : error de reconstrucción = ‖r − U_j U_jᵀ r‖ (lo que queda del residuo
#              al usar solo las j primeras componentes). RCE_0 = ‖r‖ = sqrt(pérdida).
#  r_int = #{j : RCE_j ≤ σ_j}   (intersección)      r_ε = #{j : RCE_j ≥ ε}   (precisión)
#  r_min = min(r_int, r_ε),  r_max = max(r_int, r_max anterior)
#  Cada iteración da DOS pasos de ANaGRAM: uno con r_max (explora) y otro con r_min (estabiliza).
#  Antes del "liftoff" (r_min ≥ r_max), si r_min no cambia se sube r_max en 1.
#
#  Reutiliza las funciones de la celda ANaGRAM (residuos, Jacobiano, line search, guardado).
#  Adaptaciones propias (no están en el paper), necesarias en este notebook:
#   - Espectro vía K = J Jᵀ (eigh) en vez de SVD de J: mismo U, σ = sqrt(λ).
#   - Rango máximo confiable r_piso = #{σ_j > piso·σ_1}: con float32, K no resuelve σ < ~3e-4·σ_1.
#   - El "codo" (FindElbow) se busca sobre log10(σ_j), j ≤ r_piso.
#   - r y J ya incluyen la normalización 1/N de cada término, así que RCE_0 = sqrt(pérdida total).
# =====================================================================

def _amstram_descomponer(model, lote, coef_pde, w, c_val, m_val, tam_chunk, dt):
    """Residuos, Jacobiano y espectro (orden decreciente) en el θ actual."""
    r, total, lf, l_phi, l_pi, lb = anagram_vector_residuos(model, *lote, coef_pde, w, c_val, m_val)
    J = anagram_jacobiano(model, *lote, coef_pde, w, c_val, m_val, tam_chunk=tam_chunk)
    gnorm = float(2.0 * tf.norm(tf.linalg.matvec(J, r, transpose_a=True)))
    Jd = tf.cast(J, dt)
    del J
    rd = tf.reshape(tf.cast(r, dt), [-1, 1])
    lam, Q = tf.linalg.eigh(Jd @ tf.transpose(Jd))
    lam = tf.reverse(tf.maximum(lam, 0.0), axis=[0])          # decreciente
    Q = tf.reverse(Q, axis=[1])
    coefs = tf.reshape(tf.transpose(Q) @ rd, [-1])            # c = Uᵀ r
    return dict(J=Jd, Q=Q, sigma=tf.sqrt(lam), c=coefs, r=r, total=total,
                lf=lf, l_phi=l_phi, l_pi=l_pi, lb=lb, gnorm=gnorm)


def amstram_errores_reconstruccion(c):
    """RCE_j para j = 0..S (Algoritmo 4): RCE_j² = Σ_{i>j} c_i²  (suma por la cola, más estable)."""
    c2 = np.square(np.asarray(c, dtype=np.float64))
    cola = np.cumsum(c2[::-1])[::-1]                         # cola[j] = Σ_{i≥j} c_i²  (0-indexado)
    return np.sqrt(np.concatenate([cola, [0.0]]))            # RCE[0] = ‖r‖, RCE[S] = 0


def amstram_codo(f):
    """FindElbow (Algoritmo 3) sobre la curva decreciente f_1..f_m; devuelve j en 1..m."""
    m = len(f)
    if m < 3:
        return m
    x = np.arange(1, m + 1, dtype=np.float64)
    n = np.array([f[-1] - f[0], x[0] - x[-1]])               # normal (horaria) a la cuerda
    s = n[0] * (x - x[0]) + n[1] * (f - f[0])
    return int(np.argmax(s)) + 1


def amstram_direccion_rango(D, k):
    """d = Jᵀ U_k Σ_k⁻² U_kᵀ r  (= V_k Σ_k⁻¹ U_kᵀ r), reteniendo las k componentes mayores."""
    k = int(k)
    Qk = D["Q"][:, :k]
    s2 = tf.square(D["sigma"][:k])
    d = tf.linalg.matvec(D["J"], tf.linalg.matvec(Qk, D["c"][:k] / s2), transpose_a=True)
    d = tf.cast(d, tf.float64)
    if not bool(tf.reduce_all(tf.math.is_finite(d))):
        d = tf.zeros_like(d)
    return d


def entrenar_amstramgram(model, loss_history, iteraciones, n_phys, n_bc, n_ic, c_val, m_val,
                         w=(1.0, 1.0, 1.0, 1.0), epsilon=10.0, num_chunks=16,
                         eps_precision=1e-3, tam_chunk=256,
                         pasos=0.5 ** np.linspace(0, 30, 31),
                         as_freq=20, stop_focus=None, print_every=10,
                         precision="float32", piso_rcond=None,
                         frac_stop_resampling=1.0, epsilon_adaptativo=False, epsilon_min=10.0,
                         paciencia_causal=20, iter_extra_tras_abrir=None,
                         ruta_csv="training_loss_history_AMStraMGRAM.csv",
                         ruta_pesos="AMStraMGRAM_weights.weights.h5"):
    """
    AMStraMGRAM para la PINN de Klein-Gordon (misma pérdida, muestreo y pesos causales que ANaGRAM).
    eps_precision: precisión objetivo ε en unidades de RCE = sqrt(pérdida). Se detiene si r_ε = 0.
    Historial en cada iteración; CSV + pesos cada print_every; guarda también si se interrumpe.
    - Causal: `epsilon` es el ε inicial. epsilon_adaptativo=True lo baja a la mitad (hasta epsilon_min)
      si el frente causal no avanza en paciencia_causal iteraciones (ControlCausal). Se registra el frente
      (tramos abiertos y peso del último tramo). iter_extra_tras_abrir=N detiene N iteraciones después de
      abrir todo el dominio (w_último > 0.99). frac_stop_resampling: deja de remuestrear desde esa fracción.
    """
    dt = tf.float32 if precision == "float32" else tf.float64
    if piso_rcond is None:
        piso_rcond = 3e-4 if precision == "float32" else 1e-7
    w = tf.constant(w, dtype=tf.float64)
    if stop_focus is None:
        stop_focus = 0.8 * iteraciones
    stop_resampling = frac_stop_resampling * iteraciones     # desde aquí el lote queda fijo (1.0 = nunca)
    ctrl = ControlCausal(epsilon, adaptativo=epsilon_adaptativo, eps_min=epsilon_min, paciencia=paciencia_causal)
    for k in ["eps_causal", "tramos_abiertos", "w_ultimo"]:
        loss_history.setdefault(k, [])
    for k in ["eta", "rango", "s_max", "t_jac", "t_dir", "t_ls",
              "eta2", "rce0", "r_int", "r_eps", "r_min", "r_max", "r_piso", "liftoff"]:
        loss_history.setdefault(k, [])

    liftoff, r_max, r_min_prev = False, None, None
    start_time = time.time()
    try:
        for it in range(1, iteraciones + 1):
            if it == 1 or (as_freq > 0 and it % as_freq == 0 and it <= stop_resampling):
                current_focus = it <= stop_focus
                x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = resample_all_focused(
                    n_phys, n_bc, n_ic, focus=current_focus)

            # (a) pesos causales congelados en θ_t (los mismos para los dos sub-pasos)
            eps_usado = ctrl.eps
            x_s, t_s, coef_pde, w_tramos = anagram_pesos_causales_frente(
                model, x_f, t_f, c_val, m_val, tf.constant(eps_usado, tf.float64), num_chunks)
            n_abiertos, w_ult, cambio_eps = ctrl.actualizar(w_tramos.numpy(), it)
            if cambio_eps:
                print(f"   ε causal -> {ctrl.eps:g} (el frente no avanzó en {paciencia_causal} iteraciones)")
            if ctrl.it_abierto == it:
                print(f"   >>> Dominio causal abierto en la iteración {it} (w_último > 0.99)")
            lote = (x_s, t_s, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc)

            # (b) espectro y errores de reconstrucción
            t0 = time.time()
            D = _amstram_descomponer(model, lote, coef_pde, w, c_val, m_val, tam_chunk, dt)
            t1 = time.time()
            sig = D["sigma"].numpy().astype(np.float64)
            rce = amstram_errores_reconstruccion(D["c"].numpy())
            r_piso = max(1, int(np.sum(sig > piso_rcond * sig[0])))
            j = np.arange(1, r_piso + 1)
            r_int = int(np.sum(rce[j] <= sig[j - 1]))           # intersección RCE vs σ
            r_eps = int(np.sum(rce[j] >= eps_precision))        # rango de precisión

            # (c) ranks adaptativos (Algoritmo 2, líneas 9-17)
            if r_max is None:
                r_max = amstram_codo(np.log10(sig[:r_piso]))
            r_min = min(r_int, r_eps)
            r_max = max(r_int, r_max)
            if not liftoff:
                if r_min >= r_max:
                    liftoff = True
                elif r_min_prev is not None and r_min == r_min_prev:
                    r_max += 1
            r_max = min(r_max, int(sig.shape[0]))
            r_min_prev = r_min
            cortes = []
            for rc in (r_max, r_min):                           # primero explora, luego estabiliza
                rc = int(np.clip(rc, 1, r_piso))
                if rc not in cortes:
                    cortes.append(rc)

            # (d) sub-paso 1 (r_max) y sub-paso 2 (r_min), cada uno con su line search
            d = amstram_direccion_rango(D, cortes[0])
            t2 = time.time()
            eta, _ = anagram_line_search(model, d, pasos, lote, coef_pde, w, c_val, m_val)
            eta2 = np.nan
            if len(cortes) > 1:
                D2 = _amstram_descomponer(model, lote, coef_pde, w, c_val, m_val, tam_chunk, dt)
                d2 = amstram_direccion_rango(D2, cortes[1])
                eta2, _ = anagram_line_search(model, d2, pasos, lote, coef_pde, w, c_val, m_val)
                del D2
            t3 = time.time()
            elapsed = time.time() - start_time
            t_loss, lf, l_phi, l_pi, lb, gnorm = D["total"], D["lf"], D["l_phi"], D["l_pi"], D["lb"], D["gnorm"]
            n_filas = int(D["r"].shape[0])
            del D

            loss_history["epoch"].append(it)
            loss_history["total"].append(t_loss.numpy())
            loss_history["pde"].append(lf.numpy())
            loss_history["phi"].append(l_phi.numpy())
            loss_history["pi"].append(l_pi.numpy())
            loss_history["bc"].append(lb.numpy())
            loss_history["lambda_phi"].append(float(w[1]))
            loss_history["lambda_pi"].append(float(w[2]))
            loss_history["lambda_bc"].append(float(w[3]))
            loss_history["gnorm"].append(gnorm)
            loss_history["time"].append(round(elapsed, 1))
            loss_history["eta"].append(eta)
            loss_history["rango"].append(cortes[0])
            loss_history["s_max"].append(float(sig[0]))
            loss_history["t_jac"].append(round(t1 - t0, 3))
            loss_history["t_dir"].append(round(t2 - t1, 3))
            loss_history["t_ls"].append(round(t3 - t2, 3))
            loss_history["eps_causal"].append(eps_usado)
            loss_history["tramos_abiertos"].append(n_abiertos)
            loss_history["w_ultimo"].append(w_ult)
            loss_history["eta2"].append(eta2)
            loss_history["rce0"].append(float(rce[0]))
            loss_history["r_int"].append(r_int)
            loss_history["r_eps"].append(r_eps)
            loss_history["r_min"].append(r_min)
            loss_history["r_max"].append(r_max)
            loss_history["r_piso"].append(r_piso)
            loss_history["liftoff"].append(int(liftoff))

            if it % print_every == 0 or it <= 3:
                print(f"AMStraMGRAM It {it}/{iteraciones} | Total={t_loss:.2e} | PDE={lf:.2e} | PHI={l_phi:.2e} | "
                      f"PI = {l_pi:.2e} | BC={lb:.2e} | RCE0={rce[0]:.2e} | r_int={r_int} r_eps={r_eps} "
                      f"-> cortes {cortes} (piso {r_piso}/{n_filas}) | liftoff={liftoff} | "
                      f"eta={eta:.1e}/{eta2:.1e} | paso1 {t2 - t0:.2f}s + LS/paso2 {t3 - t2:.2f}s | "
                      f"ε={eps_usado:g} frente {n_abiertos}/{num_chunks} (w_últ={w_ult:.2f}) | Time: {elapsed:.1f}s")
                _guardar_historial_anagram(loss_history, ruta_csv)
                model.save_weights(ruta_pesos)

            if (iter_extra_tras_abrir is not None and ctrl.it_abierto is not None
                    and it - ctrl.it_abierto >= iter_extra_tras_abrir):
                print(f">>> Detenido: {iter_extra_tras_abrir} iteraciones después de abrir el dominio causal.")
                break

            if r_eps == 0:
                print(f">>> AMStraMGRAM: precisión objetivo alcanzada (RCE_1 < ε = {eps_precision:.1e}) en la iteración {it}.")
                break

    except KeyboardInterrupt:
        print(f"\n>>> AMStraMGRAM interrumpido en la iteración {it}. Guardando historial y pesos...")

    _guardar_historial_anagram(loss_history, ruta_csv)
    model.save_weights(ruta_pesos)
    print(f"Historial AMStraMGRAM -> {ruta_csv} | pesos -> {ruta_pesos}")
    return x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc


## **2d. iEF (Fisher empírico mejorado)**

Wu, Yu, Zhang & Woodland, NeurIPS 2024 (arXiv:2406.06420), ec. (8) y Algoritmo 1. Reutiliza las funciones de la celda 2b; no modifica Adam, ANaGRAM ni AMStraMGRAM.

In [ ]:
# =====================================================================
#  iEF — "An Improved Empirical Fisher Approximation for Natural Gradient Descent"
#  Wu, Yu, Zhang & Woodland, NeurIPS 2024 (arXiv:2406.06420). Ec. (8) y Algoritmo 1.
#
#  Cada fila de la pérdida es una "muestra" n del modelo compuesto (D,B)∘u (como en ANaGRAM):
#     salida z_n = r_n  (residuo ya ponderado: causal + pesos w),   pérdida l_n = r_n²
#     ∇θ l_n = 2 r_n J_n ,   s_n = ‖∂l_n/∂z_n‖² = 4 r_n²
#  Actualización iEF:
#     d = ∇θlᵀ (∇θl ∇θlᵀ + λ I)⁻¹ s ,   θ ← θ − η d
#  Para mínimos cuadrados el paper muestra (Apéndice A.2) que iEF = Gauss-Newton = gradiente natural:
#  sin amortiguar (λ→0), d = 2·J⁺ r, la MISMA dirección que ANaGRAM. Lo que cambia es la regularización:
#  en vez de cortar valores singulares (rcond), amortigua cada fila con λ/(4 r_n²); los puntos que ya
#  están bien ajustados casi no mueven θ y el paso se concentra en los de residuo grande.
#
#  Reutiliza las funciones de la celda ANaGRAM (residuos, Jacobiano, pesos causales con frente,
#  ControlCausal, line search, guardado). No modifica nada de Adam, ANaGRAM ni AMStraMGRAM.
# =====================================================================

def ief_direccion(J, r, damping=1e-6, precision="float32"):
    """
    Dirección iEF (ec. 8). damping es RELATIVO: λ = damping · max diag(∇θl ∇θlᵀ).
    K = J Jᵀ se calcula en `precision` (lo caro); el sistema S×S se resuelve en float64 (Cholesky).
    Devuelve d (P,) en float64, número de filas "activas" (4 r_n² K_nn > λ) y λ.
    """
    S = int(J.shape[0])
    dt = tf.float32 if precision == "float32" else tf.float64
    Jd = tf.cast(J, dt)
    K = tf.cast(Jd @ tf.transpose(Jd), tf.float64)               # J Jᵀ  (S×S)
    r64 = tf.reshape(tf.cast(r, tf.float64), [-1])

    G = 4.0 * r64[:, None] * K * r64[None, :]                     # ∇θl ∇θlᵀ = diag(2r) K diag(2r)
    s = 4.0 * tf.square(r64)                                      # s_iEF
    diagG = tf.linalg.diag_part(G)
    lam = damping * tf.reduce_max(diagG)
    lam = tf.where(lam > 0.0, lam, tf.constant(damping, tf.float64))
    A = G + lam * tf.eye(S, dtype=tf.float64)

    try:
        chol = tf.linalg.cholesky(A)
        ok = bool(tf.reduce_all(tf.math.is_finite(chol)))       # en GPU un fallo devuelve NaN
    except tf.errors.InvalidArgumentError:                       # en CPU un fallo lanza error
        ok = False
    if ok:
        a = tf.reshape(tf.linalg.cholesky_solve(chol, s[:, None]), [-1])
    else:                                                         # respaldo si Cholesky falla
        a = tf.reshape(tf.linalg.lstsq(A, s[:, None], fast=False), [-1])

    coef = tf.cast(2.0 * r64 * a, dt)                             # ∇θlᵀ a = Jᵀ (2 r ⊙ a)
    d = tf.cast(tf.linalg.matvec(Jd, coef, transpose_a=True), tf.float64)
    if not bool(tf.reduce_all(tf.math.is_finite(d))):
        d = tf.zeros_like(d)
    activas = int(tf.reduce_sum(tf.cast(diagG > lam, tf.int32)))
    return d, activas, float(lam)


def _aplicar_paso_fijo(model, d_flat, eta):
    """θ ← θ − η d (paso fijo, como el Algoritmo 1 del paper)."""
    for v, dv in zip(model.trainable_variables, _desaplanar(model, d_flat)):
        v.assign(v - tf.cast(eta, v.dtype) * dv)


def entrenar_ief(model, loss_history, iteraciones, n_phys, n_bc, n_ic, c_val, m_val,
                 w=(1.0, 1.0, 1.0, 1.0), epsilon=10.0, num_chunks=16,
                 damping=1e-6, eta_fijo=None, tam_chunk=256,
                 pasos=0.5 ** np.linspace(0, 30, 31),
                 as_freq=1, stop_focus=None, print_every=10, precision="float32",
                 frac_stop_resampling=1.0, epsilon_adaptativo=False, epsilon_min=10.0,
                 paciencia_causal=20, iter_extra_tras_abrir=None,
                 ruta_csv="training_loss_history_iEF.csv",
                 ruta_pesos="iEF_weights.weights.h5"):
    """
    iEF para la PINN de Klein-Gordon. Mismo muestreo, pérdida, causal training (con frente y ε adaptativo)
    y guardado que ANaGRAM; solo cambia la dirección (ief_direccion) y el paso:
      eta_fijo=None -> line search en `pasos` (como ANaGRAM) | eta_fijo=η -> paso fijo (Algoritmo 1).
    """
    w = tf.constant(w, dtype=tf.float64)
    if stop_focus is None:
        stop_focus = 0.8 * iteraciones
    stop_resampling = frac_stop_resampling * iteraciones
    ctrl = ControlCausal(epsilon, adaptativo=epsilon_adaptativo, eps_min=epsilon_min, paciencia=paciencia_causal)
    for k in ["eta", "rango", "s_max", "t_jac", "t_dir", "t_ls", "eps_causal", "tramos_abiertos",
              "w_ultimo", "lambda_ief"]:
        loss_history.setdefault(k, [])

    start_time = time.time()
    try:
        for it in range(1, iteraciones + 1):
            if it == 1 or (as_freq > 0 and it % as_freq == 0 and it <= stop_resampling):
                current_focus = it <= stop_focus
                x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = resample_all_focused(
                    n_phys, n_bc, n_ic, focus=current_focus)

            # (a) pesos causales congelados en θ_t (+ frente causal)
            eps_usado = ctrl.eps
            x_s, t_s, coef_pde, w_tramos = anagram_pesos_causales_frente(
                model, x_f, t_f, c_val, m_val, tf.constant(eps_usado, tf.float64), num_chunks)
            n_abiertos, w_ult, cambio_eps = ctrl.actualizar(w_tramos.numpy(), it)
            if cambio_eps:
                print(f"   ε causal -> {ctrl.eps:g} (el frente no avanzó en {paciencia_causal} iteraciones)")
            if ctrl.it_abierto == it:
                print(f"   >>> Dominio causal abierto en la iteración {it} (w_último > 0.99)")
            lote = (x_s, t_s, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc)

            # (b) residuos y Jacobiano
            t0 = time.time()
            r, t_loss, lf, l_phi, l_pi, lb = anagram_vector_residuos(model, *lote, coef_pde, w, c_val, m_val)
            J = anagram_jacobiano(model, *lote, coef_pde, w, c_val, m_val, tam_chunk=tam_chunk)
            gnorm = float(2.0 * tf.norm(tf.linalg.matvec(J, r, transpose_a=True)))

            # (c) dirección iEF y (d) paso
            t1 = time.time()
            d, activas, lam = ief_direccion(J, r, damping=damping, precision=precision)
            del J
            t2 = time.time()
            if eta_fijo is None:
                eta, _ = anagram_line_search(model, d, pasos, lote, coef_pde, w, c_val, m_val)
            else:
                eta = float(eta_fijo)
                _aplicar_paso_fijo(model, d, eta)
            t3 = time.time()
            elapsed = time.time() - start_time

            loss_history["epoch"].append(it)
            loss_history["total"].append(t_loss.numpy())
            loss_history["pde"].append(lf.numpy())
            loss_history["phi"].append(l_phi.numpy())
            loss_history["pi"].append(l_pi.numpy())
            loss_history["bc"].append(lb.numpy())
            loss_history["lambda_phi"].append(float(w[1]))
            loss_history["lambda_pi"].append(float(w[2]))
            loss_history["lambda_bc"].append(float(w[3]))
            loss_history["gnorm"].append(gnorm)
            loss_history["time"].append(round(elapsed, 1))
            loss_history["eta"].append(eta)
            loss_history["rango"].append(activas)          # filas activas (no amortiguadas)
            loss_history["s_max"].append(np.nan)
            loss_history["t_jac"].append(round(t1 - t0, 3))
            loss_history["t_dir"].append(round(t2 - t1, 3))
            loss_history["t_ls"].append(round(t3 - t2, 3))
            loss_history["eps_causal"].append(eps_usado)
            loss_history["tramos_abiertos"].append(n_abiertos)
            loss_history["w_ultimo"].append(w_ult)
            loss_history["lambda_ief"].append(lam)

            if it % print_every == 0 or it <= 3:
                print(f"iEF It {it}/{iteraciones} | Total={t_loss:.2e} | PDE={lf:.2e} | PHI={l_phi:.2e} | "
                      f"PI = {l_pi:.2e} | BC={lb:.2e} | eta={eta:.1e} | activas={activas}/{int(r.shape[0])} | "
                      f"λ={lam:.1e} | gnorm ={gnorm:.2e} | paso: J {t1 - t0:.2f}s + dir {t2 - t1:.2f}s + "
                      f"LS {t3 - t2:.2f}s | ε={eps_usado:g} frente {n_abiertos}/{num_chunks} (w_últ={w_ult:.2f}) | "
                      f"Time: {elapsed:.1f}s")
                _guardar_historial_anagram(loss_history, ruta_csv)
                model.save_weights(ruta_pesos)

            if (iter_extra_tras_abrir is not None and ctrl.it_abierto is not None
                    and it - ctrl.it_abierto >= iter_extra_tras_abrir):
                print(f">>> Detenido: {iter_extra_tras_abrir} iteraciones después de abrir el dominio causal.")
                break

    except KeyboardInterrupt:
        print(f"\n>>> iEF interrumpido en la iteración {it}. Guardando historial y pesos...")

    _guardar_historial_anagram(loss_history, ruta_csv)
    model.save_weights(ruta_pesos)
    print(f"Historial iEF -> {ruta_csv} | pesos -> {ruta_pesos}")
    return x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc


## **2e. SPRING (gradiente natural estocástico con momentum, "tipo Adam")**

Goldshlager, Abrahamsen & Lin (2024); usado en PINNs por Guzmán-Cordero et al. (NeurIPS 2025). Requiere ejecutar antes las celdas 2b (ANaGRAM) y 2d (iEF). No modifica los demás optimizadores.

In [ ]:
# =====================================================================
#  SPRING — gradiente natural estocástico con momentum ("un gradiente natural tipo Adam")
#  Goldshlager, Abrahamsen & Lin (2024), "A Kaczmarz-inspired approach to accelerate the
#  optimization of neural network wavefunctions" (SPRING); usado en PINNs por
#  Guzmán-Cordero et al., NeurIPS 2025 ("Improving ENGD through Woodbury, Momentum, and Randomization").
#
#  Idea (lo que hace funcionar a Adam, pero con la geometría del gradiente natural):
#   - lotes nuevos en cada paso (ruido de datos)          -> as_freq = 1
#   - pasos PEQUEÑOS y fijos en vez de line search         -> eta_fijo (no ajusta el lote a la perfección)
#   - momentum: acumula direcciones de muchos lotes        -> mu
#   - opcional: ruido en los parámetros con forma G⁺       -> temperatura (dinámica de Langevin riemanniana)
#
#  Dirección SPRING (resuelta en el espacio de filas S×S, como la ruta "ntk" de ANaGRAM):
#     d_k = μ d_{k-1} + Jᵀ (J Jᵀ + λ I)⁻¹ (r − μ J d_{k-1}) ,     θ ← θ − η d_k
#  Con μ = 0 y λ → 0 es exactamente ANaGRAM (d = J⁺ r).
#  Ruido (opcional): θ ← θ + sqrt(2 η T) ξ ,  ξ = Jᵀ U Λ⁻¹ z ,  z ~ N(0, I)  =>  Cov(ξ) = (JᵀJ)⁺
#
#  Reutiliza las funciones de la celda ANaGRAM (residuos, Jacobiano, causal con frente, guardado).
#  No modifica Adam, ANaGRAM, AMStraMGRAM ni iEF.
# =====================================================================

def spring_direccion(J, r, d_prev, mu=0.9, damping=1e-6, piso_rcond=1e-3, precision="float32",
                     temperatura=0.0, eta=0.1, rng=None):
    """
    Devuelve (d_k, ruido, rango). damping es relativo a λ_max(J Jᵀ). piso_rcond descarta
    componentes con σ < piso·σ_max (ruido numérico; 1e-3 en float32, ~1e-7 en float64).
    """
    dt = tf.float32 if precision == "float32" else tf.float64
    Jd = tf.cast(J, dt)
    rd = tf.reshape(tf.cast(r, dt), [-1])
    lam, Q = tf.linalg.eigh(Jd @ tf.transpose(Jd))               # J Jᵀ = Q Λ Qᵀ
    lam_max = lam[-1]
    keep = lam > (piso_rcond ** 2) * lam_max
    lam_d = damping * lam_max
    inv = tf.where(keep, 1.0 / (tf.where(keep, lam, tf.ones_like(lam)) + lam_d), tf.zeros_like(lam))

    if d_prev is None:
        v = rd
    else:
        v = rd - mu * tf.linalg.matvec(Jd, tf.cast(d_prev, dt))   # r − μ J d_{k-1}
    w = tf.linalg.matvec(Q, inv * tf.linalg.matvec(Q, v, transpose_a=True))
    d = tf.cast(tf.linalg.matvec(Jd, w, transpose_a=True), tf.float64)
    if d_prev is not None:
        d = d + mu * d_prev
    if not bool(tf.reduce_all(tf.math.is_finite(d))):
        d = tf.zeros_like(d) if d_prev is None else tf.cast(d_prev, tf.float64)

    ruido = None
    if temperatura > 0.0:
        z = rng.standard_normal(int(lam.shape[0]))
        inv_sig = tf.where(keep, 1.0 / tf.sqrt(tf.where(keep, lam, tf.ones_like(lam))), tf.zeros_like(lam))
        # ξ = Jᵀ Q Λ⁻¹ z sobre las componentes retenidas  ->  Cov = V Σ⁻² Vᵀ = (JᵀJ)⁺
        xi = tf.linalg.matvec(Jd, tf.linalg.matvec(Q, (inv_sig ** 2) * tf.constant(z, dt)), transpose_a=True)
        ruido = tf.cast(np.sqrt(2.0 * eta * temperatura) * xi, tf.float64)
    rango = int(tf.reduce_sum(tf.cast(keep, tf.int32)))
    return d, ruido, rango


def entrenar_spring(model, loss_history, iteraciones, n_phys, n_bc, n_ic, c_val, m_val,
                    w=(1.0, 1.0, 1.0, 1.0), epsilon=10.0, num_chunks=16,
                    eta_fijo=0.1, mu=0.9, damping=1e-6, piso_rcond=1e-3, precision="float32",
                    temperatura=0.0, frac_enfriamiento=0.7, tam_chunk=256,
                    pasos=0.5 ** np.linspace(0, 30, 31),
                    as_freq=1, stop_focus=None, print_every=10,
                    frac_stop_resampling=1.0, epsilon_adaptativo=False, epsilon_min=10.0,
                    paciencia_causal=20, iter_extra_tras_abrir=None, semilla_ruido=0,
                    ruta_csv="training_loss_history_SPRING.csv",
                    ruta_pesos="SPRING_weights.weights.h5"):
    """
    SPRING para la PINN de Klein-Gordon. Mismo muestreo, pérdida, causal (con frente y ε adaptativo)
    y guardado que ANaGRAM. eta_fijo=None -> line search (como ANaGRAM); número -> paso fijo.
    temperatura T: ruido de Langevin con covarianza (JᵀJ)⁺, enfriado linealmente hasta 0 en
    frac_enfriamiento·iteraciones (0 = sin ruido).
    """
    w = tf.constant(w, dtype=tf.float64)
    if stop_focus is None:
        stop_focus = 0.8 * iteraciones
    stop_resampling = frac_stop_resampling * iteraciones
    ctrl = ControlCausal(epsilon, adaptativo=epsilon_adaptativo, eps_min=epsilon_min, paciencia=paciencia_causal)
    rng = np.random.default_rng(semilla_ruido)
    for k in ["eta", "rango", "s_max", "t_jac", "t_dir", "t_ls", "eps_causal", "tramos_abiertos",
              "w_ultimo", "temperatura", "norma_d", "ruido_rel"]:
        loss_history.setdefault(k, [])

    d_prev = None
    start_time = time.time()
    try:
        for it in range(1, iteraciones + 1):
            if it == 1 or (as_freq > 0 and it % as_freq == 0 and it <= stop_resampling):
                current_focus = it <= stop_focus
                x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = resample_all_focused(
                    n_phys, n_bc, n_ic, focus=current_focus)

            eps_usado = ctrl.eps
            x_s, t_s, coef_pde, w_tramos = anagram_pesos_causales_frente(
                model, x_f, t_f, c_val, m_val, tf.constant(eps_usado, tf.float64), num_chunks)
            n_abiertos, w_ult, cambio_eps = ctrl.actualizar(w_tramos.numpy(), it)
            if cambio_eps:
                print(f"   ε causal -> {ctrl.eps:g} (el frente no avanzó en {paciencia_causal} iteraciones)")
            if ctrl.it_abierto == it:
                print(f"   >>> Dominio causal abierto en la iteración {it} (w_último > 0.99)")
            lote = (x_s, t_s, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc)

            t0 = time.time()
            r, t_loss, lf, l_phi, l_pi, lb = anagram_vector_residuos(model, *lote, coef_pde, w, c_val, m_val)
            J = anagram_jacobiano(model, *lote, coef_pde, w, c_val, m_val, tam_chunk=tam_chunk)
            gnorm = float(2.0 * tf.norm(tf.linalg.matvec(J, r, transpose_a=True)))

            t1 = time.time()
            T_k = temperatura * max(0.0, 1.0 - it / max(1.0, frac_enfriamiento * iteraciones))
            eta_ruido = eta_fijo if eta_fijo is not None else 0.1
            d, ruido, rango = spring_direccion(J, r, d_prev, mu=mu, damping=damping, piso_rcond=piso_rcond,
                                               precision=precision, temperatura=T_k, eta=eta_ruido, rng=rng)
            del J
            t2 = time.time()
            if eta_fijo is None:
                eta, _ = anagram_line_search(model, d, pasos, lote, coef_pde, w, c_val, m_val)
            else:
                eta = float(eta_fijo)
                _aplicar_paso_fijo(model, d, eta)
            ruido_rel = 0.0
            if ruido is not None:
                _aplicar_paso_fijo(model, -ruido, 1.0)                     # θ ← θ + ruido
                ruido_rel = float(tf.norm(ruido)) / max(eta * float(tf.norm(d)), 1e-300)
            d_prev = d
            t3 = time.time()
            elapsed = time.time() - start_time

            loss_history["epoch"].append(it)
            loss_history["total"].append(t_loss.numpy())
            loss_history["pde"].append(lf.numpy())
            loss_history["phi"].append(l_phi.numpy())
            loss_history["pi"].append(l_pi.numpy())
            loss_history["bc"].append(lb.numpy())
            loss_history["lambda_phi"].append(float(w[1]))
            loss_history["lambda_pi"].append(float(w[2]))
            loss_history["lambda_bc"].append(float(w[3]))
            loss_history["gnorm"].append(gnorm)
            loss_history["time"].append(round(elapsed, 1))
            loss_history["eta"].append(eta)
            loss_history["rango"].append(rango)
            loss_history["s_max"].append(np.nan)
            loss_history["t_jac"].append(round(t1 - t0, 3))
            loss_history["t_dir"].append(round(t2 - t1, 3))
            loss_history["t_ls"].append(round(t3 - t2, 3))
            loss_history["eps_causal"].append(eps_usado)
            loss_history["tramos_abiertos"].append(n_abiertos)
            loss_history["w_ultimo"].append(w_ult)
            loss_history["temperatura"].append(T_k)
            loss_history["norma_d"].append(float(tf.norm(d)))
            loss_history["ruido_rel"].append(ruido_rel)

            if it % print_every == 0 or it <= 3:
                print(f"SPRING It {it}/{iteraciones} | Total={t_loss:.2e} | PDE={lf:.2e} | PHI={l_phi:.2e} | "
                      f"PI = {l_pi:.2e} | BC={lb:.2e} | eta={eta:.1e} μ={mu} T={T_k:.1e} (ruido/paso={ruido_rel:.2f}) | rango={rango}/{int(r.shape[0])} | "
                      f"‖d‖={float(tf.norm(d)):.2e} | paso: J {t1 - t0:.2f}s + dir {t2 - t1:.2f}s + LS {t3 - t2:.2f}s | "
                      f"ε={eps_usado:g} frente {n_abiertos}/{num_chunks} (w_últ={w_ult:.2f}) | Time: {elapsed:.1f}s")
                _guardar_historial_anagram(loss_history, ruta_csv)
                model.save_weights(ruta_pesos)

            if (iter_extra_tras_abrir is not None and ctrl.it_abierto is not None
                    and it - ctrl.it_abierto >= iter_extra_tras_abrir):
                print(f">>> Detenido: {iter_extra_tras_abrir} iteraciones después de abrir el dominio causal.")
                break

    except KeyboardInterrupt:
        print(f"\n>>> SPRING interrumpido en la iteración {it}. Guardando historial y pesos...")

    _guardar_historial_anagram(loss_history, ruta_csv)
    model.save_weights(ruta_pesos)
    print(f"Historial SPRING -> {ruta_csv} | pesos -> {ruta_pesos}")
    return x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc


### **Selección del optimizador**

`OPTIMIZADOR = "ADAM"` corre exactamente el entrenamiento original (Adam + LRA + causal). `"ANaGRAM"` usa corte fijo `rcond`. `"AMStraMGRAM"` elige el corte en cada iteración. `"iEF"` usa el Fisher empírico mejorado (Wu et al., 2024). `"SPRING"` es gradiente natural estocástico con momentum (pasos pequeños, tipo Adam).

In [ ]:
# =====================  SELECCIÓN DEL OPTIMIZADOR  =====================
OPTIMIZADOR = "SPRING"  # "ADAM" | "ANaGRAM" | "AMStraMGRAM" | "iEF" | "SPRING"

# --- Hiperparámetros de ANaGRAM (solo se usan si OPTIMIZADOR == "ANaGRAM") ---
# Una iteración de ANaGRAM NO equivale a una época de Adam: cada iteración resuelve un
# mínimos cuadrados (Gauss-Newton) + line search. Es ~100-1000x más cara, pero se usan ~50x menos.
iter_anagram        = 400                     # el paper usa 1000-4000
n_phys_anagram      = 2000                     # puntos PDE por paso (Adam usa N = 12000)
n_ic_anagram        = n_phys_anagram // 2      # misma proporción que Adam (n_ic = N // 2)
n_bc_anagram        = n_phys_anagram // 4      # misma proporción que Adam (n_bc = N // 4)
#   filas del Jacobiano: S = n_phys + 2·n_ic + 2·n_bc = 5000  -> costo por paso ~ O(S²·P)
as_freq_anagram     = 0                        # remuestreo cada cuántos pasos (1 = como Adam; 0 = lote fijo como en el paper)
w_anagram           = (1.0, 50.0, 50.0, 1.0)     # pesos FIJOS [PDE, φ, π, BC] = w_tensor inicial de Adam (sin LRA)
rcond_anagram       = 1e-6                     # corte de la pseudo-inversa: σ ≤ rcond·σ_max -> 0. Con 1e-6 el line search
                                               # elige η ~ 1e-8 y se estanca (direcciones mal condicionadas); 1e-3 da η ~ 0.1-0.5
precision_anagram   = "float32"                # álgebra lineal (K = JJᵀ, eigh): "float32" (T4/L4: float64 es ~30x más lento) | "float64" (A100)
metodo_anagram      = "ntk"                    # "ntk" (S×S, RECOMENDADO si S<P) | "svd" (Algoritmo 2 literal, muy lento en GPU) | "engd" (P×P)
tam_chunk_anagram   = 256                      # puntos por bloque al calcular J (subir a 512-1024 si sobra memoria GPU)
pasos_anagram       = 0.5 ** np.linspace(0, 30, 31)   # rejilla del line search (igual que el repo ENGD)
print_every_anagram = 10                       # imprime y guarda CSV + pesos cada N iteraciones (el historial se registra en todas)

# --- Hiperparámetros de AMStraMGRAM (solo si OPTIMIZADOR == "AMStraMGRAM") ---
# Usa los mismos n_*_anagram, w_anagram, tam_chunk_anagram y pasos_anagram de arriba.
# No usa rcond: el rango de corte se elige solo en cada iteración. Cada iteración da 2 pasos (~2x ANaGRAM).
iter_amstram        = 300                      # iteraciones (cada una = hasta 2 pasos de ANaGRAM)
epsilon_amstram     = 1e-6                     # precisión objetivo ε en unidades de sqrt(pérdida): 1e-3 <-> pérdida 1e-6
as_freq_amstram     = 50                       # remuestreo cada N iteraciones (el método sigue la RCE sobre el mismo lote; 1 = cada paso)
precision_amstram   = "float32"                # "float32" (T4) | "float64" (más rango utilizable, ~4x más lento en T4)
piso_rcond_amstram  = None                     # None = automático: 3e-4 (float32) / 1e-7 (float64). Rango máx = #{σ > piso·σ_max}

# --- Causal training para ANaGRAM / AMStraMGRAM (no afecta a Adam, que usa `epsilon` de la celda de entrenamiento) ---
epsilon_anagram        = 1000.0    # ε causal INICIAL. Con ε = 10 los pesos causales casi no frenan a ANaGRAM
epsilon_adaptativo     = True      # baja ε a la mitad si el frente causal no avanza en `paciencia_causal` iteraciones
epsilon_min_anagram    = 10.0      # nunca por debajo de este valor (el de Adam)
paciencia_causal       = 20        # iteraciones sin que avance el frente antes de bajar ε
iter_extra_tras_abrir  = None      # p. ej. 100: detener 100 iteraciones después de abrir todo el dominio (w_último > 0.99)
frac_stop_resampling_anagram = 1.0 # fracción de las iteraciones desde la que el lote queda fijo (1.0 = nunca)

# --- Hiperparámetros de iEF (solo si OPTIMIZADOR == "iEF") ---
# Usa n_*_anagram, w_anagram, tam_chunk_anagram, pasos_anagram y el bloque causal de arriba.
iter_ief            = 400
damping_ief         = 1e-6        # λ relativo a max diag(∇l ∇lᵀ). Filas con 4r²K_nn < λ quedan "congeladas"
eta_ief             = None        # None = line search (como ANaGRAM) | número = paso fijo (Algoritmo 1 del paper)
precision_ief       = "float32"   # precisión de K = J Jᵀ; el sistema S×S siempre se resuelve en float64
as_freq_ief         = 1           # remuestreo cada N iteraciones

# --- Hiperparámetros de SPRING (solo si OPTIMIZADOR == "SPRING"): gradiente natural estocástico "tipo Adam" ---
# Usa n_*_anagram, w_anagram, tam_chunk_anagram, pasos_anagram y el bloque causal de arriba.
iter_spring         = 600
eta_spring          = 0.1         # paso FIJO pequeño (None = line search como ANaGRAM). 1.0 = paso completo de Gauss-Newton
mu_spring           = 0.0         # momentum. Paso efectivo ≈ eta/(1-mu): con mu=0.9 usa eta ~0.01-0.02 (en la prueba corta no mejoró)
damping_spring      = 1e-6        # λ relativo a λ_max(J Jᵀ) (amortiguamiento de Tikhonov)
piso_rcond_spring   = 1e-3        # descarta σ < piso·σ_max (ruido numérico en float32)
precision_spring    = "float32"
temperatura_spring  = 0.0         # ruido de Langevin con cov. (JᵀJ)⁺ (0 = apagado). Escala típica 1e-8–1e-7: mira 'ruido/paso' en el log (~0.1–1)
frac_enfriamiento_spring = 0.7    # la temperatura llega a 0 en esta fracción de las iteraciones
as_freq_spring      = 1           # lotes nuevos en cada paso (como Adam)


In [ ]:
#Seed
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Parámetros de ejecución
neurons = 64
num_blocks = 6
epochs_adam = 100000
max_iter_lbfgs = 0
print_every = 1000
as_freq = 1
stop_resampling = epochs_adam
n_phys = N
n_bc = N // 4
n_ic = N // 2

#LRA
alpha_lra = 0.9
w_tensor = tf.Variable([1.0, 1.0, 1.0, 1e-1], dtype=tf.float64)

#SIREN
omega_0 = 15.0

#Causal Training
epsilon = 10.0
num_chunks = 16

# Instanciar modelo
model = PINN(
    num_layers=num_blocks,
    hidden_dim=neurons,
    use_fourier=True,          # True = activa el Embedding | False = lo desactiva
    num_fourier_features=256,
    sigma=1.0
)

# Forzar inicialización construyendo las capas
x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = resample_all_focused(n_phys, n_bc, n_ic, focus=True)
_ = model([x_f, t_f])

# Optimizador
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3, decay_steps=2000, decay_rate=0.9, staircase=False
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
optimizer.build(model.trainable_variables)

# Historial principal con "gnorm" inicializado correctamente
loss_history = {"epoch": [], "total": [], "pde": [], "phi": [], "pi": [] ,"bc": [], "lambda_phi": [], "lambda_pi":[], "lambda_bc": [], "gnorm": [], "time": []}

# Funciones de pérdida

c_tf = tf.cast(c, tf.float64)
m_tf = tf.cast(m, tf.float64)

# Muestreo inicial
x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = resample_all_focused(n_phys, n_bc, n_ic, focus=True)

start_time = time.time()

if OPTIMIZADOR == "ADAM":
    print(" FASE 1: Exploración Global con Adam + LRA")

    stop_focus = epochs_adam * 0.8

    for epoch in range(1, epochs_adam + 1):

        if epoch % as_freq == 0 and epoch <= stop_resampling:
            # Usar muestreo enfocado al principio, muestreo global al final
            current_focus = True if epoch <= stop_focus else False

            x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = resample_all_focused(
                n_phys, n_bc, n_ic, focus=current_focus
            )

        t_loss, lf, l_phi, l_pi, lb, gnorm = train_step(
            model, optimizer, x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc, c_tf, m_tf, w_tensor = w_tensor, alpha_lra = alpha_lra, epsilon = epsilon, num_chunks = num_chunks
        )

        if epoch % print_every == 0 or epoch == 1:
            elapsed = time.time() - start_time
            print(f"AdamW Ep {epoch}/{epochs_adam} | Total={t_loss:.2e} | PDE={lf:.2e} | PHI={l_phi:.2e} | PI = {l_pi:.2e} | BC={lb:.2e} | w_phi={w_tensor[1]:.2e} | w_pi={w_tensor[2]:.2e} | w_bc={w_tensor[3]:.2e} | gnorm ={gnorm:.2e} | LR={lr_schedule(optimizer.iterations):.2e} | Time: {elapsed:.1f}s")

            # Guardado del histórico para AdamW
            loss_history["epoch"].append(epoch)
            loss_history["total"].append(t_loss.numpy())
            loss_history["pde"].append(lf.numpy())
            loss_history["phi"].append((l_phi).numpy())
            loss_history["pi"].append((l_pi).numpy())
            loss_history["bc"].append(lb.numpy())
            loss_history["lambda_phi"].append(w_tensor[1].numpy())
            loss_history["lambda_pi"].append(w_tensor[2].numpy())
            loss_history["lambda_bc"].append(w_tensor[3].numpy())
            loss_history["gnorm"].append(gnorm.numpy())
            loss_history["time"].append(round(elapsed, 1))


elif OPTIMIZADOR == "ANaGRAM":
    P_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
    S_filas = n_phys_anagram + 2 * n_ic_anagram + 2 * n_bc_anagram
    print(f" FASE 1: ANaGRAM | P = {P_params} parámetros | S = {S_filas} filas | "
          f"J ≈ {S_filas * P_params * 8 / 1e9:.2f} GB (float64)")

    x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = entrenar_anagram(
        model, loss_history, iter_anagram,
        n_phys_anagram, n_bc_anagram, n_ic_anagram, c_tf, m_tf,
        w=w_anagram, epsilon=epsilon_anagram, num_chunks=num_chunks,
        rcond=rcond_anagram, metodo=metodo_anagram, tam_chunk=tam_chunk_anagram,
        pasos=pasos_anagram, as_freq=as_freq_anagram, print_every=print_every_anagram,
        precision=precision_anagram,
        epsilon_adaptativo=epsilon_adaptativo, epsilon_min=epsilon_min_anagram,
        paciencia_causal=paciencia_causal, iter_extra_tras_abrir=iter_extra_tras_abrir,
        frac_stop_resampling=frac_stop_resampling_anagram
    )

elif OPTIMIZADOR == "AMStraMGRAM":
    P_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
    S_filas = n_phys_anagram + 2 * n_ic_anagram + 2 * n_bc_anagram
    print(f" FASE 1: AMStraMGRAM | P = {P_params} parámetros | S = {S_filas} filas | "
          f"J ≈ {S_filas * P_params * 8 / 1e9:.2f} GB (float64)")

    x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = entrenar_amstramgram(
        model, loss_history, iter_amstram,
        n_phys_anagram, n_bc_anagram, n_ic_anagram, c_tf, m_tf,
        w=w_anagram, epsilon=epsilon_anagram, num_chunks=num_chunks,
        eps_precision=epsilon_amstram, tam_chunk=tam_chunk_anagram,
        pasos=pasos_anagram, as_freq=as_freq_amstram, print_every=print_every_anagram,
        precision=precision_amstram, piso_rcond=piso_rcond_amstram,
        epsilon_adaptativo=epsilon_adaptativo, epsilon_min=epsilon_min_anagram,
        paciencia_causal=paciencia_causal, iter_extra_tras_abrir=iter_extra_tras_abrir,
        frac_stop_resampling=frac_stop_resampling_anagram
    )

elif OPTIMIZADOR == "iEF":
    P_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
    S_filas = n_phys_anagram + 2 * n_ic_anagram + 2 * n_bc_anagram
    print(f" FASE 1: iEF | P = {P_params} parámetros | S = {S_filas} filas | "
          f"J ≈ {S_filas * P_params * 8 / 1e9:.2f} GB (float64)")

    x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = entrenar_ief(
        model, loss_history, iter_ief,
        n_phys_anagram, n_bc_anagram, n_ic_anagram, c_tf, m_tf,
        w=w_anagram, epsilon=epsilon_anagram, num_chunks=num_chunks,
        damping=damping_ief, eta_fijo=eta_ief, tam_chunk=tam_chunk_anagram,
        pasos=pasos_anagram, as_freq=as_freq_ief, print_every=print_every_anagram,
        precision=precision_ief,
        epsilon_adaptativo=epsilon_adaptativo, epsilon_min=epsilon_min_anagram,
        paciencia_causal=paciencia_causal, iter_extra_tras_abrir=iter_extra_tras_abrir,
        frac_stop_resampling=frac_stop_resampling_anagram
    )

elif OPTIMIZADOR == "SPRING":
    P_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
    S_filas = n_phys_anagram + 2 * n_ic_anagram + 2 * n_bc_anagram
    print(f" FASE 1: SPRING | P = {P_params} parámetros | S = {S_filas} filas | "
          f"J ≈ {S_filas * P_params * 8 / 1e9:.2f} GB (float64)")

    x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc = entrenar_spring(
        model, loss_history, iter_spring,
        n_phys_anagram, n_bc_anagram, n_ic_anagram, c_tf, m_tf,
        w=w_anagram, epsilon=epsilon_anagram, num_chunks=num_chunks,
        eta_fijo=eta_spring, mu=mu_spring, damping=damping_spring, piso_rcond=piso_rcond_spring,
        precision=precision_spring, temperatura=temperatura_spring,
        frac_enfriamiento=frac_enfriamiento_spring, tam_chunk=tam_chunk_anagram,
        pasos=pasos_anagram, as_freq=as_freq_spring, print_every=print_every_anagram,
        epsilon_adaptativo=epsilon_adaptativo, epsilon_min=epsilon_min_anagram,
        paciencia_causal=paciencia_causal, iter_extra_tras_abrir=iter_extra_tras_abrir,
        frac_stop_resampling=frac_stop_resampling_anagram
    )

else:
    raise ValueError('OPTIMIZADOR debe ser "ADAM", "ANaGRAM", "AMStraMGRAM", "iEF" o "SPRING"')

if max_iter_lbfgs == 0:
  pass
else:
  print("\n FASE 2: Refinamiento Fino con L-BFGS")
  lbfgs_iter = 0
  initial_weights = get_flat_weights(model)


  results = scipy.optimize.minimize(
      fun=lbfgs_objective,
      x0=initial_weights,
      args=(x_f, t_f, x_ic, t_ic, phi0, pi0, x_bc_l, x_bc_r, t_bc, c_tf, m_tf, epsilon, num_chunks),
      method='L-BFGS-B',
      jac=True,
      options={
          'maxiter': max_iter_lbfgs,
          'maxfun': max_iter_lbfgs,
          'ftol': 1.0 * np.finfo(float).eps,
          'gtol': 1e-8
      }
  )
  set_flat_weights(model, tf.convert_to_tensor(results.x, dtype=tf.float64))
  print(f"\nL-BFGS Finalizado: {results.message}")
  print(f"Pérdida final alcanzada: {results.fun:.2e}")

# GUARDADO
valid_history = {k: v for k, v in loss_history.items() if len(v) > 0}
min_len = min([len(v) for v in valid_history.values()])
clean_history = {k: v[:min_len] for k, v in valid_history.items()}
df_loss = pd.DataFrame(clean_history)
df_loss.to_csv("training_loss_history_AbuPINN.csv", index=False)

# Guardamos los pesos del modelo
model.save_weights("AbuPINN_ultimate_weights.weights.h5")

print(f"Entrenamiento finalizado exitosamente y guardado (Filas guardadas: {min_len}).")

In [ ]:
import matplotlib.pyplot as plt

# Gráfica de Convergencia
plt.figure(figsize=(10, 5))

# Paleta de tonos pastel enfocada en aqua, verdes y azules
color_total  = "#000000" # Negro intenso
color_pde    = "#E63946" # Rojo carmesí vibrante
color_ic_phi = "#2A9D8F" # Verde esmeralda sólido
color_ic_pi  = "#F4A261" # Naranja fuerte
color_bc     = "#4361EE" # Azul rey puro

plt.semilogy(loss_history["epoch"], loss_history["total"], color=color_total, label="Total Loss", linewidth=1)
plt.semilogy(loss_history["epoch"], loss_history["pde"], color=color_pde, label="PDE Loss (Física)", linestyle="--", alpha=0.9, linewidth=1)
plt.semilogy(loss_history["epoch"], loss_history["phi"], color=color_ic_phi, label="IC Loss phi (Inicial)", linestyle="--", alpha=0.9, linewidth=1)
plt.semilogy(loss_history["epoch"], loss_history["phi"], color=color_ic_pi, label="IC Loss pi (Inicial)", linestyle="--", alpha=0.9, linewidth=1)
plt.semilogy(loss_history["epoch"], loss_history["bc"], color=color_bc, label="BC Loss (Frontera)", linestyle="--", alpha=0.9, linewidth=1)

plt.title("Historial de Entrenamiento PINN", fontsize=14, pad=15)
plt.xlabel("Épocas", fontsize=11)
plt.ylabel("Pérdida", fontsize=11)

# Estilizar la leyenda para que se vea más limpia
plt.legend(frameon=True, facecolor='#ffffff', edgecolor='none', fontsize=10)

# Suavizar la cuadrícula y darle un fondo sutil a los ejes para resaltar los pasteles
plt.grid(True, which="both", ls="--", alpha=0.5, color='#E0E0E0')
plt.gca().set_facecolor('#FAFAFA')

# Quitar los bordes superior y derecho para un look más moderno
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_color('#DDDDDD')
plt.gca().spines['bottom'].set_color('#DDDDDD')

plt.show()

In [ ]:
# Nombre correcto basado en tu captura de archivos: AbuPINN (sin 'n' extra)
nombre_archivo = f"ANaGRAM_weights.weights.h5"
ruta_completa = os.path.join("/content", nombre_archivo)

if os.path.exists(ruta_completa):
    try:
        model.load_weights(ruta_completa)
        print(f"   -> ¡ÉXITO! Pesos cargados desde: {nombre_archivo}")
    except Exception as e:
        print(f"\n[ERROR CRÍTICO]: El archivo existe pero no pudo ser leído: {e}")
else:
    print(f"\n[ERROR]: El archivo '{nombre_archivo}' NO existe en /content/ABUN_PINN/")
    print("Archivos encontrados en la carpeta:")
    print(os.listdir("/content/ABUN_PINN/"))
    # No uses sys.exit() aquí si quieres continuar con pesos aleatorios
    print("   -> Procediendo con inicialización aleatoria.")

In [ ]:
# Parámetros físicos
sigma = 1.0
k0_sim = [1.0, 2.0, 5.0]
A_sim = [1.0, 1.0, 1.0]
k_boost = 0.0
m = 1.0
c = 1.0

# Parámetros numéricos
N = 4096                    # puntos espaciales
L = 80 * sigma              # tamaño del dominio
T_max = L/2 - 4*sigma       # tiempo final (evita que el frente toque el borde)
n_t = 200                   # snapshots a guardar

# ============== MALLADO ==============
x = np.linspace(-L/2, L/2, N, endpoint=False)
dx = x[1] - x[0]
k = 2 * np.pi * np.fft.fftfreq(N, d=dx)

# Frecuencia angular de Klein-Gordon: cada modo k oscila a ω_k = √(c²k² + m²)
omega_k = np.sqrt(c**2 * k**2 + m**2)

# ============== CONDICIÓN INICIAL ==============
# Envolvente gaussiana modulada por suma de modos cosenoidales
k0_arr = np.array(k0_sim).reshape(-1, 1)
A_arr = np.array(A_sim).reshape(-1, 1)
x_row = x.reshape(1, -1)

fase = np.sum(A_arr * (np.exp(1j * k0_arr * x_row) + np.exp(-1j * k0_arr * x_row)), axis=0)
psi_static = np.exp(-x**2 / (2 * sigma**2)) * fase
psi_moving = psi_static * np.exp(1j * k_boost * x)
psi_moving = psi_moving / np.sqrt(np.max(np.abs(psi_moving)**2))

phi0 = psi_moving.real
phi0 = phi0 / np.sqrt(np.max(np.abs(phi0)**2))

# π₀ = ∂φ/∂t|_{t=0} consistente con la relación de dispersión KG
phi_k_aux = np.fft.fft(psi_moving)
pi_k_aux = -1j * omega_k * phi_k_aux
pi0 = np.fft.ifft(pi_k_aux).real

# ============== PROPAGACIÓN EXACTA ==============
phi_k_0 = np.fft.fft(phi0)
pi_k_0 = np.fft.fft(pi0)

t_save = np.linspace(0.0, T_max, n_t)
Z_exact = np.zeros((n_t, N))

# Manejo del límite ω_k → 0 (solo ocurre si m=0 y k=0)
small = np.isclose(omega_k, 0.0)
omega_safe = np.where(small, 1.0, omega_k)

for i, t in enumerate(t_save):
    cos_wt = np.cos(omega_k * t)
    sin_wt = np.sin(omega_k * t)
    sin_wt_div_w = sin_wt / omega_safe
    sin_wt_div_w[small] = t  # lim_{ω→0} sin(ωt)/ω = t

    # Propagación exacta de cada modo de Fourier
    phi_k_t = phi_k_0 * cos_wt + pi_k_0 * sin_wt_div_w
    Z_exact[i, :] = np.fft.ifft(phi_k_t).real


In [ ]:
# 1. Parámetros del dominio (deben coincidir con el solver pseudo-espectral)
sigma = 1.0
L = 80 * sigma
X_min, X_max = -L/2, L/2
T_min, T_max = 0.0, L/2 - 4*sigma
N = 4096    # Puntos espaciales de la FFT (debe coincidir con el solver)
n_t = 200   # Snapshots almacenados

# 2. Generar la malla de prueba
# IMPORTANTE: endpoint=False en x para respetar la periodicidad de la FFT
x_test_phys = np.linspace(X_min, X_max, N, endpoint=False)
t_test_phys = np.linspace(T_min, T_max, n_t)
X_grid, T_grid = np.meshgrid(x_test_phys, t_test_phys)

# Z_exact debe haber sido calculado previamente con el solver pseudo-espectral
# (shape (n_t, N))

# 3. Predicción del modelo ABU-PINN
X_input = tf.cast(X_grid.reshape(-1, 1), tf.float64)
T_input = tf.cast(T_grid.reshape(-1, 1), tf.float64)

pred_out = model([X_input, T_input]).numpy()
Z_pred = pred_out.flatten().reshape(n_t, N)

# 4. Error Relativo L2
error_l2 = np.linalg.norm(Z_exact - Z_pred, ord='fro') / np.linalg.norm(Z_exact, ord='fro')
print(f"\n>>> Error Relativo L2 vs Referencia Pseudo-Espectral: {error_l2:.4e} <<<")

# 5. Graficar
extent = [X_min, X_max, T_min, T_max]
vmax = max(np.abs(Z_exact).max(), np.abs(Z_pred).max())

fig, axs = plt.subplots(1, 3, figsize=(20, 5))
cmap = "jet"

# Solución de referencia
im1 = axs[0].imshow(Z_exact, extent=extent, aspect='auto', origin='lower',
                    cmap=cmap)
axs[0].set_title("Solución Pseudo-Espectral (Referencia)")
axs[0].set_xlabel("x")
axs[0].set_ylabel("t")
plt.colorbar(im1, ax=axs[0])

# Predicción de la PINN
im2 = axs[1].imshow(Z_pred, extent=extent, aspect='auto', origin='lower',
                    cmap=cmap)
axs[1].set_title(f"PINN (Error L2: {error_l2:.2e})")
axs[1].set_xlabel("x")
plt.colorbar(im2, ax=axs[1])

# Error absoluto
im3 = axs[2].imshow(np.abs(Z_exact - Z_pred), extent=extent, aspect='auto',
                    origin='lower', cmap='inferno')
axs[2].set_title(r"Error Absoluto $|\phi_{exact} - \phi_{pred}|$")
axs[2].set_xlabel("x")
plt.colorbar(im3, ax=axs[2])

plt.tight_layout()
plt.show()

In [ ]:
# Guardar el modelo completo de Keras
modelo_path = '/content/ABUN_PINN/PINN_model.keras'
model.save(modelo_path)
print(f"Modelo completo guardado exitosamente en: {modelo_path}")

In [ ]:
import pandas as pd
import numpy as np

# Calcular el error absoluto medio (Mean Absolute Error)
error_abs = np.mean(np.abs(Z_exact - Z_pred))

# Número de parámetros del modelo
num_params = model.count_params()

# Función de costo promedio de las última capa
costo = df_loss['total'].iloc[-1]

# Crear la tabla de errores ampliada
df_errores = pd.DataFrame(
    {
        "Error Relativo L2": [f"{error_l2:.2f}"],
        "Error Absoluto": [f"{error_abs:.4f}"],
        "Número de parámetros": [f"{num_params}"],
        "Costo final": [f"{costo:.2e}"]
    },
    index=["PINN"]
)

# Mostrar la tabla
df_errores

In [ ]:
# Guardar la tabla de errores en un archivo CSV
csv_path = '/content/ABUN_PINN/errores_PINN.csv'
df_errores.to_csv(csv_path, index=True)
print(f"Tabla de errores guardada exitosamente en: {csv_path}")